# LlamaIndex Agentic RAG (SEC v2 Parity)

This notebook ports the LangGraph SEC v2 state machine to a LlamaIndex event-driven Workflow.

Parity goals:
- Same retrieval stack: BM25 + Dense (Chroma) + RRF + CrossEncoder rerank.
- Same control flow: planner -> retriever -> context evaluator -> generator -> critic -> repair/retry.
- Same model strings and temperatures as the LangGraph notebook.
- No re-embedding: connect to existing Chroma collection via ChromaVectorStore.

In [1]:
# Install/repair dependencies in the active notebook kernel if needed.
# %pip install -U chromadb python-dotenv rank-bm25 sentence-transformers pydantic
# %pip install -U llama-index-core llama-index-vector-stores-chroma
# %pip install -U llama-index-llms-groq llama-index-llms-gemini

In [2]:
import os
import sys
import re
import json
import time
import asyncio
import threading
from dataclasses import dataclass
from pathlib import Path
from collections import deque
from typing import Any, Dict, List, Optional, TypedDict, Literal

import numpy as np
import pandas as pd
import chromadb

from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from pydantic import BaseModel, Field, field_validator, model_validator

from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.workflow import Workflow, Event, StartEvent, StopEvent, step, Context

from llama_index.llms.groq import Groq as LIGroq
from llama_index.llms.google_genai import GoogleGenAI as LIGoogleGenAI

# ========== IMPORT FROM SHARED MODULES ==========
# Resolve repo root robustly from notebook location.
CWD = Path.cwd()

def detect_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "config.py").exists() and (p / "shared_retriever.py").exists():
            return p
    for p in [start, *start.parents]:
        if (p / ".env").exists():
            return p
    return start

PROJECT_ROOT = detect_project_root(CWD)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Path helper used by drift/ingestion cells

def resolve_path_from_env(path_str: str) -> Path:
    p = Path(path_str)
    if p.exists():
        return p
    if not p.is_absolute():
        c1 = (CWD / p).resolve()
        if c1.exists():
            return c1
        c2 = (PROJECT_ROOT / p).resolve()
        if c2.exists():
            return c2
    return p

# Load .env and import shared modules
load_dotenv(PROJECT_ROOT / ".env")

from config import CONFIG, resolve_model_name, resolve_fallback_model_names, get_provider_order
from shared_retriever import initialize_corpus, CorpusIndex

print(f"Ã¢Å“â€œ Imported CONFIG from {PROJECT_ROOT / 'config.py'}")
print(f"Ã¢Å“â€œ Imported shared_retriever from {PROJECT_ROOT / 'shared_retriever.py'}")

c:\Users\jeeey\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Loading embedding model: nomic-ai/nomic-embed-text-v1.5...


<All keys matched successfully>
c:\Users\jeeey\GenAI-with-LLMs\shared_retriever.py:139: RuntimeWarning: trust_remote_code=True for dense model 'nomic-ai/nomic-embed-text-v1.5'. Only enable this for trusted model repositories.
  warnings.warn(


Loading reranker model: cross-encoder/ms-marco-MiniLM-L-6-v2...
Ã¢Å“â€œ Imported CONFIG from c:\Users\jeeey\GenAI-with-LLMs\config.py
Ã¢Å“â€œ Imported shared_retriever from c:\Users\jeeey\GenAI-with-LLMs\shared_retriever.py


In [3]:
# ========== ENVIRONMENT AUDIT: API Key Verification ==========
# Non-interactive API key loading for notebook automation (mirrors LangGraph v4)
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "").strip()
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "").strip()
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "").strip()
ACTIVE_GEMINI_API_KEY = GOOGLE_API_KEY or GEMINI_API_KEY

if not ACTIVE_GEMINI_API_KEY:
    print("[WARN] Neither GOOGLE_API_KEY nor GEMINI_API_KEY is set. Gemini LLM calls will fail later.")
elif GOOGLE_API_KEY:
    print("✓ GOOGLE_API_KEY loaded from environment.")
else:
    print("✓ GEMINI_API_KEY loaded from environment; notebook will reuse it for Gemini calls.")

if GROQ_API_KEY:
    print("✓ GROQ_API_KEY loaded from environment.")


✓ GEMINI_API_KEY loaded from environment; notebook will reuse it for Gemini calls.
✓ GROQ_API_KEY loaded from environment.


In [4]:
# ========== ADAPTER: Bridge shared_retriever output to LlamaIndex-compatible formats ==========
# shared_retriever returns Dict; LlamaIndex may expect RetrievedChunk or NodeWithScore
# Define adapter class for compatibility with LlamaIndex workflow

@dataclass
class RetrievedChunk:
    """Adapter dataclass matching shared_retriever output schema."""
    doc_name: str
    company: str
    page_num: int
    chunk_id: str
    raw_chunk: str
    contextual_chunk: str
    score: float
    source: str
    ticker: Optional[str] = None


def dict_to_retrieved_chunk(d: Dict[str, Any]) -> RetrievedChunk:
    """Convert shared_retriever Dict output to RetrievedChunk adapter."""
    meta = d.get('metadata', {})
    return RetrievedChunk(
        doc_name=meta.get('doc_name', ''),
        company=meta.get('company', ''),
        page_num=meta.get('page_num', 0),
        chunk_id=d.get('chunk_id', ''),
        raw_chunk=d.get('raw_text', ''),
        contextual_chunk=d.get('text', ''),
        score=float(d.get('score', 0.0)),
        source=d.get('source', ''),
        ticker=meta.get('ticker', None),
    )


def extract_retrieved_doc_names(chunks: List[RetrievedChunk]) -> List[str]:
    return list(dict.fromkeys([c.doc_name for c in chunks]))


def cleanup_planner_query(
    query: Optional[str],
    ticker: Optional[str] = None,
    filing_year: Optional[int] = None,
    form_type: Optional[str] = None,
) -> str:
    """Clean and augment planner query with metadata hints."""
    cleaned = (query or "").strip()
    if not cleaned:
        cleaned = "financial metric from SEC filing"
    upper = cleaned.upper()
    if upper.startswith("SELECT ") and " FROM " in upper:
        cleaned = cleaned.replace("SELECT", "").replace("FROM filings", "").strip()
    for token in ("WHERE", "FROM", "AND", "=", "'", '"'):
        cleaned = cleaned.replace(token, " ")
    cleaned = cleaned.replace("filing year", "filing_year")
    cleaned = cleaned.replace("|", " ")
    cleaned = cleaned.replace("_", " ")
    cleaned = cleaned.replace("company", "")
    cleaned = cleaned.replace("ticker", "")
    cleaned = " ".join(cleaned.split())

    parts = [cleaned]
    if ticker and ticker not in cleaned:
        parts.insert(0, ticker)
    if filing_year and str(filing_year) not in cleaned:
        parts.append(str(filing_year))
    if form_type and form_type not in cleaned:
        parts.append(form_type)
    return " ".join([p for p in parts if p]).strip()


def fails_retrieval_sanity_check(
    question: str, 
    sub_queries: List[Dict[str, Any]], 
    retrieved: List[RetrievedChunk]
) -> bool:
    """Check if retrieved chunks match expected companies/periods from planner."""
    if not question.strip():
        return False
    if not CONFIG["enable_retrieval_sanity_check"]:
        return False
    if not retrieved:
        return False
    expected_tickers = {sq.get("ticker") for sq in sub_queries if sq.get("ticker")}
    if not expected_tickers:
        return False
    present_tickers = {c.ticker for c in retrieved if c.ticker}
    if not present_tickers:
        return False
    return not expected_tickers.issubset(present_tickers)


# ========== INITIALIZE SHARED CORPUS ==========
print("Initializing shared CorpusIndex from shared_retriever.py...")
global_index = initialize_corpus(
    chunks_jsonl=CONFIG["sec_chunks_path"],
    chroma_db_path=CONFIG["chroma_db_path"],
)
print(f"Ã¢Å“â€œ Corpus initialized: {global_index.df.shape[0]:,} chunks, "
      f"{global_index.chroma_collection.count():,} vectors")
print(f"Ã¢Å“â€œ Adjacent chunk expansion: ENABLED")
print(f"Ã¢Å“â€œ Dense model: {CONFIG['dense_model_name']}")


Initializing shared CorpusIndex from shared_retriever.py...
Loading chunks from ..\sec_rag_team_share\sec_data\chunks\sec_chunks.jsonl...
Loaded 13275 chunks
Building BM25 index...
Connecting to Chroma DB at ..\sec_rag_team_share\chroma_db...
Chroma collection: 588 vectors
CorpusIndex ready.
Ã¢Å“â€œ Corpus initialized: 13,275 chunks, 588 vectors
Ã¢Å“â€œ Adjacent chunk expansion: ENABLED
Ã¢Å“â€œ Dense model: nomic-ai/nomic-embed-text-v1.5


In [5]:
# ========== RECOVERY: Reinitialize corpus if needed ==========
print("[Recovery] Reinitializing shared CorpusIndex...")
global_index = initialize_corpus(
    chunks_jsonl=CONFIG["sec_chunks_path"],
    chroma_db_path=CONFIG["chroma_db_path"],
)
print(f"[Recovery] Ã¢Å“â€œ Corpus ready: {global_index.df.shape[0]:,} chunks")


[Recovery] Reinitializing shared CorpusIndex...
Loading chunks from ..\sec_rag_team_share\sec_data\chunks\sec_chunks.jsonl...
Loaded 13275 chunks
Building BM25 index...
Connecting to Chroma DB at ..\sec_rag_team_share\chroma_db...
Chroma collection: 588 vectors
CorpusIndex ready.
[Recovery] Ã¢Å“â€œ Corpus ready: 13,275 chunks


In [6]:
# ========== CONFIG AUGMENTATION: Prior Advanced Results Reuse ==========
# Add reuse_prior_advanced_results toggle and path (mirror LangGraph v4)

# Default: Enable CSV reuse, point to baseline_advanced_rag/results/advanced_results.csv
CONFIG.setdefault("reuse_prior_advanced_results", True)
CONFIG.setdefault(
    "prior_advanced_results_path",
    str((PROJECT_ROOT / "baseline_advanced_rag" / "results" / "advanced_results.csv").resolve())
)

print(f"✓ Prior advanced-results reuse: {CONFIG['reuse_prior_advanced_results']} | path={CONFIG['prior_advanced_results_path']}")


✓ Prior advanced-results reuse: True | path=C:\Users\jeeey\GenAI-with-LLMs\baseline_advanced_rag\results\advanced_results.csv


In [7]:
# ========== PRIOR RESULTS LOADER: Cache CSV for reuse during evaluation ==========

def _normalize_prior_question_id(value: Any) -> str:
    """Normalize question IDs to SEC_NNN format for matching."""
    text = str(value).strip()
    if not text or text.lower() == "nan":
        return ""
    try:
        return f"SEC_{int(float(text)):03d}"
    except Exception:
        return text


def load_prior_advanced_results(eval_df: pd.DataFrame) -> Dict[str, Any]:
    """
    Load prior advanced_results.csv and create lookup indexes by id and question text.
    Falls back to live retrieval if toggle is False, file not found, or empty.
    """
    if not CONFIG.get("reuse_prior_advanced_results", True):
        print("✓ Prior advanced-results reuse disabled; LlamaIndex will run live retrieval.")
        return {}

    # Build candidate paths: config path → results_dir → baseline_advanced_rag default
    candidate_paths = []
    configured_path = str(CONFIG.get("prior_advanced_results_path", "")).strip()
    if configured_path:
        candidate_paths.append(Path(configured_path))
    candidate_paths.append(Path(CONFIG["results_dir"]) / "advanced_results.csv")
    candidate_paths.append(PROJECT_ROOT / "baseline_advanced_rag" / "results" / "advanced_results.csv")

    # Find first existing path
    existing_paths = []
    seen = set()
    for candidate in candidate_paths:
        try:
            resolved = candidate.expanduser().resolve()
        except Exception:
            resolved = candidate.expanduser()
        key = str(resolved).lower()
        if key in seen:
            continue
        seen.add(key)
        if resolved.exists():
            existing_paths.append(resolved)

    if not existing_paths:
        message = (
            "No prior advanced_results.csv found. LlamaIndex will fall back to live retrieval."
        )
        if CONFIG.get("require_prior_advanced_results", False):
            raise FileNotFoundError(message)
        print(f"ℹ {message}")
        return {}

    selected_path = existing_paths[0]
    prior_df = pd.read_csv(selected_path).copy()
    
    if prior_df.empty:
        message = f"Prior advanced results at {selected_path} are empty; falling back to live retrieval."
        if CONFIG.get("require_prior_advanced_results", False):
            raise ValueError(message)
        print(f"ℹ {message}")
        return {}

    # Normalize CSV columns for matching
    if "predicted_answer" not in prior_df.columns and "final_answer" in prior_df.columns:
        prior_df["predicted_answer"] = prior_df["final_answer"]
    
    prior_df["normalized_id"] = prior_df.get("question_id", pd.Series(index=prior_df.index, dtype=object)).apply(_normalize_prior_question_id)
    prior_df["question_normalized"] = prior_df.get("question", pd.Series(index=prior_df.index, dtype=object)).fillna("").astype(str).str.strip()

    # Build lookup dictionaries
    by_id = {}
    by_question = {}
    for _, row in prior_df.iterrows():
        norm_id = row.get("normalized_id", "")
        if norm_id:
            by_id[norm_id] = row
        norm_q = row.get("question_normalized", "")
        if norm_q:
            by_question[norm_q] = row

    # Compute match statistics
    eval_ids = eval_df["id"].fillna("").astype(str).str.strip() if "id" in eval_df.columns else pd.Series([""] * len(eval_df))
    eval_questions = eval_df["question"].fillna("").astype(str).str.strip() if "question" in eval_df.columns else pd.Series([""] * len(eval_df))
    matched_by_id = int(eval_ids.isin(set(by_id.keys())).sum())
    matched_by_question = int(eval_questions.isin(set(by_question.keys())).sum())

    print(
        f"✓ Loaded prior advanced results: {selected_path} | rows={len(prior_df)} | "
        f"matches_by_id={matched_by_id}/{len(eval_df)} | matches_by_question={matched_by_question}/{len(eval_df)}"
    )

    return {
        "path": str(selected_path),
        "df": prior_df,
        "by_id": by_id,
        "by_question": by_question,
    }


def get_prior_advanced_row(eval_row: pd.Series, prior_cache: Dict[str, Any]) -> Optional[pd.Series]:
    """
    Look up evaluation question in prior cache by id first, then by question text.
    Returns prior row if found, None otherwise.
    """
    if not prior_cache:
        return None

    qid = str(eval_row.get("id", "")).strip()
    if qid and qid in prior_cache.get("by_id", {}):
        return prior_cache["by_id"][qid]

    question = str(eval_row.get("question", "")).strip()
    if question and question in prior_cache.get("by_question", {}):
        return prior_cache["by_question"][question]

    return None


def normalize_answer_text(answer: Any) -> str:
    """Normalize answer text for consistent output."""
    if answer is None:
        return ""
    text = str(answer).strip()
    # Clean up extra whitespace
    text = " ".join(text.split())
    return text


def build_advanced_output_from_prior(eval_row: pd.Series, prior_row: pd.Series, prior_cache_path: str) -> Dict[str, Any]:
    """
    Build LlamaIndex output dict from prior CSV row, preserving column mappings.
    Maps: final_answer → final_answer, llm_correctness_score → llm_correctness_score, etc.
    """
    answer = normalize_answer_text(
        prior_row.get("predicted_answer", prior_row.get("final_answer", ""))
    )
    rewritten_query = normalize_answer_text(
        prior_row.get("rewritten_query", eval_row.get("question", ""))
    )

    # Map prior CSV columns to output dict
    n_chunks = prior_row.get("n_chunks")
    retrieved_chunks = []  # Prior CSV doesn't contain actual chunks, just metadata
    
    # Extract judgment scores from CSV
    correctness = prior_row.get("llm_correctness_score")
    claims_covered = prior_row.get("llm_claims_covered")
    faithfulness = prior_row.get("llm_faithfulness_score")

    latency_sec = prior_row.get("latency_sec", prior_row.get("latency_seconds", None))
    correctness_score = None if pd.isna(correctness) else float(correctness)
    claims_covered_score = None if pd.isna(claims_covered) else float(claims_covered)
    faithfulness_score = None if pd.isna(faithfulness) else float(faithfulness)

    return {
        "final_answer": answer,
        "generated_answer": answer,
        "rewritten_query": rewritten_query,
        "retrieved_chunks": retrieved_chunks,
        "retrieved_doc_names": [],
        "judge_score": correctness_score,
        "judge_claims_covered": claims_covered_score,
        "judge_reason": str(prior_row.get("llm_correctness_reason", "Loaded from prior advanced results")).strip(),
        "llm_correctness_score": correctness_score,
        "llm_claims_covered": claims_covered_score,
        "llm_correctness_reason": str(prior_row.get("llm_correctness_reason", "Loaded from prior advanced results")).strip(),
        "llm_faithfulness_score": faithfulness_score,
        "llm_faithfulness_reason": str(prior_row.get("llm_faithfulness_reason", "Loaded from prior advanced results")).strip(),
        "latency_sec": None if pd.isna(latency_sec) else float(latency_sec),
        "output_source": "prior_advanced_results",
        "prior_advanced_results_path": prior_cache_path,
        "precomputed_n_chunks": None if pd.isna(n_chunks) else int(n_chunks),
        "needs_decomposition": False,  # Prior results are already processed
        "critic_decision": "accept",  # Prior results bypassed critic
        "repair_used": False,  # No repair needed for prior results
    }


In [8]:
class SubQuery(BaseModel):
    query: Optional[str] = Field(default=None, description="Search-optimized sub-query text for retrieval")
    ticker: Optional[str] = Field(default=None, description="Company ticker if company-specific")
    filing_year: Optional[int] = Field(default=None, description="Calendar year when the filing was submitted")
    form_type: Optional[str] = Field(default=None, description="10-K or 10-Q if specified")

    @field_validator("query", mode="before")
    @classmethod
    def normalize_query(cls, v):
        if isinstance(v, dict):
            parts: List[str] = []
            for key in ("query", "query_field", "metric", "line_item", "table", "topic"):
                value = v.get(key)
                if value is not None and str(value).strip():
                    parts.append(str(value).strip())
            fields = v.get("fields")
            if isinstance(fields, list) and fields:
                parts.append(" ".join(str(item).strip() for item in fields if str(item).strip()))
            return " | ".join(dict.fromkeys(parts)) if parts else None
        return v

    @field_validator("ticker", mode="before")
    @classmethod
    def normalize_ticker(cls, v):
        if v is None:
            return None
        cleaned = str(v).strip().upper()
        return cleaned or None

    @field_validator("form_type", mode="before")
    @classmethod
    def normalize_form_type(cls, v):
        if v is None:
            return None
        cleaned = str(v).strip().upper()
        return cleaned if cleaned in {"10-K", "10-Q"} else None


class PlannerOutput(BaseModel):
    needs_decomposition: bool = Field(
        description="True when the question requires multiple distinct filings, periods, or companies."
    )
    sub_queries: List[SubQuery] = Field(
        description="One sub-query for single-filing questions, otherwise 2-3 sub-queries split by company or period."
    )

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        if not isinstance(data, dict):
            return data
        normalized = dict(data)
        if "sub_queries" not in normalized:
            for alias in ("queries", "sub_query", "query"):
                if alias in normalized:
                    value = normalized[alias]
                    normalized["sub_queries"] = value if isinstance(value, list) else [value]
                    break
        if "needs_decomposition" not in normalized:
            sub_queries = normalized.get("sub_queries", [])
            normalized["needs_decomposition"] = isinstance(sub_queries, list) and len(sub_queries) > 1
        return normalized

    @field_validator("needs_decomposition", mode="before")
    @classmethod
    def coerce_bool(cls, v):
        if isinstance(v, str):
            return v.strip().lower() in {"true", "1", "yes"}
        return bool(v)


class ContextEvalOutput(BaseModel):
    relevant: bool
    reason: str


class CriticOutput(BaseModel):
    decision: Literal["accept", "repair", "insufficient_data"]
    reason: str


class RepairOutput(BaseModel):
    decision: Literal["accept", "warn", "refuse"]
    revised_answer: str
    needs_new_retrieval: bool
    reason: str


def _coerce_fraction_01(value: Any, default: float = 0.0) -> float:
    if value is None:
        return default
    if isinstance(value, bool):
        return 1.0 if value else 0.0
    if isinstance(value, (int, float)):
        return max(0.0, min(1.0, float(value)))
    if isinstance(value, list):
        for item in value:
            try:
                return _coerce_fraction_01(item, default=default)
            except Exception:
                continue
        return default
    text = str(value).strip()
    if not text:
        return default
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    if not match:
        return default
    number = float(match.group(0))
    if number > 1.0:
        number = number / 100.0 if number <= 100.0 else 1.0
    return max(0.0, min(1.0, number))


class JudgeOutput(BaseModel):
    score: float
    claims_covered: float
    reason: str

    @field_validator("score", "claims_covered", mode="before")
    @classmethod
    def coerce_fraction_fields(cls, v):
        return _coerce_fraction_01(v, default=0.0)


PLANNER_SYSTEM = (
    "You are a financial research planner working with SEC filings from these companies:\n"
    "AAPL (Apple), BAC (Bank of America), BRK (Berkshire Hathaway), COST (Costco), "
    "CVX (Chevron), JNJ (Johnson & Johnson), JPM (JPMorgan Chase), MSFT (Microsoft), "
    "NVDA (NVIDIA), UNH (UnitedHealth), WMT (Walmart), XOM (ExxonMobil).\n\n"
    "Decide whether the question requires data from multiple distinct filings, then produce sub-queries.\n\n"
    "Decompose (needs_decomposition=True) when the question:\n"
    "  Ã¢â‚¬Â¢ Explicitly compares two different fiscal years (e.g. 2023 vs 2024)\n"
    "  Ã¢â‚¬Â¢ Compares two different companies\n\n"
    "Do NOT decompose single-period, single-company questions.\n\n"
    "For each sub-query set ticker if company-specific, filing_year if year-specific.\n"
    "Every sub-query object must include a non-empty query field.\n"
    "filing_year = the calendar year the 10-K or 10-Q was filed "
    "(e.g. Apple FY2024 10-K was filed in November 2024, so filing_year=2024). "
    "Return a valid JSON object only that matches the requested schema."
)

CONTEXT_EVAL_SYSTEM = (
    "You judge whether retrieved context is relevant and sufficient to answer a financial question. "
    "Mark as not relevant only if the context is clearly from the wrong company or period, or completely empty. "
    "Partial tables and incomplete passages still count as relevant if they contain the right metric. "
    "Return valid JSON only with keys relevant and reason."
)

GENERATOR_SYSTEM = (
    "You are a financial analyst answering questions using only the retrieved filing context. "
    "Be precise with numbers, units, fiscal periods, and line-item names. "
    "Do not invent or infer a filing form, filing date, filing title, source document, or 'latest' claim unless it is explicitly supported by the retrieved context. "
    "Do not mention a form like 10-K or 10-Q, a filing date, or a quoted source unless it appears in the context. "
    "If context supports only part of the answer, provide only the supported part and explicitly note what is missing. "
    "If the context does not contain the answer, say so explicitly rather than estimating."
)

CRITIC_SYSTEM = (
    "You are a critic for a financial RAG system. Evaluate the answer and choose one decision: "
    "accept, repair, or insufficient_data. Use repair when the data is present but the answer used it incorrectly. "
    "Use insufficient_data only when the required financial data is absent from the retrieved context. "
    "Return valid JSON only with keys decision and reason."
)

REPAIR_SYSTEM = (
    "You repair a financial RAG answer after critique. Default to accept with a revised answer whenever possible. "
    "Pay close attention to fiscal year, quarter, units, sign, and line-item name. "
    "Remove any claim about filing form, filing date, source document, document recency, or section title unless that exact detail is explicitly present in the retrieved context. "
    "If the answer is incomplete, grounded to the wrong source, or cites an unsupported filing/date/document, set needs_new_retrieval=true. "
    "If context supports only a subset of the answer, keep only the supported subset and say what is missing. "
    "Return valid JSON only with keys decision, revised_answer, needs_new_retrieval, and reason."
)

JUDGE_CORRECTNESS_SYSTEM = (
    "Score the candidate answer against the reference answer from 0 to 1 for a financial QA task. "
    "1 = correct value, correct units, correct period. "
    "0 = wrong number, wrong company, wrong period, or completely missing. "
    "Give partial credit when the answer is directionally right but incomplete. "
    "Also set claims_covered: fraction of distinct facts/numbers in the reference present in the candidate. "
    "Return valid JSON only with keys score, claims_covered, and reason."
)


def _strip_code_fence(text: str) -> str:
    t = (text or "").strip()
    if t.startswith("```"):
        t = re.sub(r"^```[a-zA-Z0-9_\-]*", "", t).strip()
        if t.endswith("```"):
            t = t[:-3].strip()
    return t


def _run_coro_sync(coro):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None

    if loop and loop.is_running():
        import nest_asyncio
        nest_asyncio.apply()
        return loop.run_until_complete(coro)
    return asyncio.run(coro)


def llm_complete(llm, prompt: str) -> str:
    timeout_sec = float(CONFIG.get("llm_request_timeout_sec", 90))
    for attempt in range(CONFIG["llm_max_retries"]):
        try:
            rate_limiter.wait()
            if hasattr(llm, "acomplete"):
                response = _run_coro_sync(asyncio.wait_for(llm.acomplete(prompt), timeout=timeout_sec))
            else:
                from concurrent.futures import ThreadPoolExecutor
                with ThreadPoolExecutor(max_workers=1) as executor:
                    response = executor.submit(llm.complete, prompt).result(timeout=timeout_sec)
            return response.text.strip()
        except Exception:
            if attempt == CONFIG["llm_max_retries"] - 1:
                raise
            time.sleep(CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt))
    raise RuntimeError("Unreachable")


def invoke_structured(llm, schema, system: str, user: str, fallback):
    prompt = (
        f"System:\n{system}\n\n"
        "Return ONLY valid JSON, with no markdown fences.\n\n"
        f"User:\n{user}"
    )
    for attempt in range(CONFIG["llm_max_retries"]):
        try:
            raw = llm_complete(llm, prompt)
            data = json.loads(_strip_code_fence(raw))
            return schema.model_validate(data)
        except Exception:
            if attempt == CONFIG["llm_max_retries"] - 1:
                return fallback
            time.sleep(CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt))
    return fallback


def invoke_text(llm, system: str, user: str, fallback_text: str = "") -> str:
    prompt = f"System:\n{system}\n\nUser:\n{user}"
    try:
        return llm_complete(llm, prompt)
    except Exception:
        return fallback_text


def llm_judge_correctness(question: str, reference_answer: str, candidate_answer: str) -> tuple[float, float, str]:
    judged = invoke_structured(
        judge_llm,
        JudgeOutput,
        JUDGE_CORRECTNESS_SYSTEM,
        (
            f"Question:\n{question}\n\n"
            f"Reference answer:\n{reference_answer}\n\n"
            f"Candidate answer:\n{candidate_answer}"
        ),
        JudgeOutput(score=0.0, claims_covered=0.0, reason="Judge fallback"),
    )
    return (
        max(0.0, min(1.0, float(judged.score))),
        max(0.0, min(1.0, float(judged.claims_covered))),
        judged.reason,
    )


def format_retrieved_context(chunks: List[RetrievedChunk], max_chunks: int, max_chars: int) -> str:
    selected = chunks[:max_chunks]
    parts: List[str] = []
    for i, c in enumerate(selected, start=1):
        parts.append(
            f"[Doc {i}] Company: {c.company} | Filing: {c.doc_name}\n"
            f"Content: {c.raw_chunk}"
        )
    joined = "\n\n".join(parts)
    return joined[:max_chars]

In [9]:
class AgentStateRequired(TypedDict):
    question: str
    # NOTE: global_index is now global, not per-state


class AgentState(AgentStateRequired, total=False):
    rewritten_query: str
    sub_queries: List[Dict[str, Any]]
    needs_decomposition: bool

    retrieved_chunks: List[Any]
    retrieved_doc_names: List[str]
    retrieval_sanity_failed: bool

    context_retries: int
    context_eval_relevant: bool

    generated_answer: str

    critic_decision: str
    critic_reason: str

    repair_used: bool
    repair_decision: str
    repair_reason: str
    needs_new_retrieval: bool
    repair_retrieval_count: int
    is_repair_retrieval: bool
    use_seeded_retrieval: bool
    seeded_from_prior_advanced: bool

    final_answer: str
    golden_answer: Optional[str]
    judge_score: Optional[float]
    judge_claims_covered: Optional[float]
    judge_reason: Optional[str]


class PlanEvent(Event):
    state: AgentState


class RetrieveEvent(Event):
    state: AgentState


class RetrievedEvent(Event):
    state: AgentState


class GenerateEvent(Event):
    state: AgentState


class CritiqueEvent(Event):
    state: AgentState


class RepairEvent(Event):
    state: AgentState


class RepairRetrieveEvent(Event):
    state: AgentState


class SecAgenticRAGWorkflow(Workflow):
    @step
    async def query_planning(self, _ctx: Context, ev: StartEvent) -> PlanEvent:
        question = ev.get("question")
        advanced_output = ev.get("advanced_output") or {}

        if not question:
            raise ValueError("Workflow requires question")

        seeded_chunks = advanced_output.get("retrieved_chunks") or []
        seeded_doc_names = advanced_output.get("retrieved_doc_names") or []
        seeded_query = normalize_answer_text(advanced_output.get("rewritten_query", question)) or question
        seeded_from_prior = advanced_output.get("output_source") == "prior_advanced_results"

        state: AgentState = {
            "question": question,
            "rewritten_query": seeded_query,
            "retrieved_chunks": seeded_chunks,
            "retrieved_doc_names": seeded_doc_names,
            "context_retries": 0,
            "repair_used": False,
            "repair_retrieval_count": 0,
            "is_repair_retrieval": False,
            "retrieval_sanity_failed": False,
            "sub_queries": [],
            "needs_decomposition": False,
            "use_seeded_retrieval": bool(seeded_chunks),
            "seeded_from_prior_advanced": seeded_from_prior,
        }

        if advanced_output:
            state["sub_queries"] = [{
                "query": seeded_query,
                "ticker": None,
                "filing_year": None,
                "form_type": None,
            }]
            state["needs_decomposition"] = False
            print(
                f"  [Planner] seeded {'prior advanced' if seeded_from_prior else 'advanced'} | 1 sub-query"
            )
            return PlanEvent(state=state)

        result = invoke_structured(
            planner_llm,
            PlannerOutput,
            PLANNER_SYSTEM,
            f"Question:\n{question}",
            PlannerOutput(needs_decomposition=False, sub_queries=[SubQuery(query=question)]),
        )

        sub_queries: List[Dict[str, Any]] = []
        for sq in result.sub_queries:
            sq_dict = sq.model_dump()
            if not sq_dict.get("query"):
                parts = [question]
                if sq_dict.get("ticker"):
                    parts.append(f"ticker {sq_dict['ticker']}")
                if sq_dict.get("filing_year"):
                    parts.append(f"filing year {sq_dict['filing_year']}")
                if sq_dict.get("form_type"):
                    parts.append(f"form {sq_dict['form_type']}")
                sq_dict["query"] = " | ".join(parts)
            sq_dict["query"] = cleanup_planner_query(
                sq_dict.get("query", ""),
                ticker=sq_dict.get("ticker"),
                filing_year=sq_dict.get("filing_year"),
                form_type=sq_dict.get("form_type"),
            )
            sub_queries.append(sq_dict)

        if not sub_queries:
            sub_queries = [{
                "query": cleanup_planner_query(question),
                "ticker": None,
                "filing_year": None,
                "form_type": None,
            }]

        state["sub_queries"] = sub_queries
        state["rewritten_query"] = sub_queries[0]["query"]
        state["needs_decomposition"] = result.needs_decomposition

        print(
            f"  [Planner] {'decomposed' if state['needs_decomposition'] else 'single'} | "
            f"{len(sub_queries)} sub-quer{'ies' if len(sub_queries) > 1 else 'y'}"
        )
        if state["needs_decomposition"]:
            for sq in sub_queries:
                print(
                    f"    -> {sq['query']}  "
                    f"(ticker={sq.get('ticker')}, year={sq.get('filing_year')}, form={sq.get('form_type')})"
                )

        # Event mapping: PlanEvent == LangGraph transition query_planner -> hybrid_retriever.
        return PlanEvent(state=state)

    @step
    async def hybrid_retriever(
        self,
        _ctx: Context,
        ev: PlanEvent | RetrieveEvent | RepairRetrieveEvent,
    ) -> RetrievedEvent:
        state = ev.state
        sub_queries = state.get("sub_queries", [])

        is_repair_retrieval = isinstance(ev, RepairRetrieveEvent)
        if is_repair_retrieval:
            state["is_repair_retrieval"] = True

        if state.get("use_seeded_retrieval", False) and state.get("retrieved_chunks") and not is_repair_retrieval:
            retrieved = state.get("retrieved_chunks", [])
        elif len(sub_queries) <= 1:
            q = state.get("rewritten_query", state["question"])
            sq = sub_queries[0] if sub_queries else {}
            # ========== CALL SHARED RETRIEVER ==========
            retrieved_dicts = global_index.hybrid_search(
                query=q,
                top_k=CONFIG["rerank_top_k"],
                ticker=sq.get("ticker"),
                filing_year=sq.get("filing_year"),
                form_type=sq.get("form_type"),
            )
            # Convert shared_retriever Dict output to RetrievedChunk adapter
            retrieved = [dict_to_retrieved_chunk(d) for d in retrieved_dicts]
        else:
            per_k = CONFIG["decomposition_top_k_per_subquery"]
            merged: Dict[str, RetrievedChunk] = {}
            for sq in sub_queries:
                # ========== CALL SHARED RETRIEVER FOR EACH SUBQUERY ==========
                retrieved_dicts = global_index.hybrid_search(
                    query=sq["query"],
                    top_k=per_k,
                    ticker=sq.get("ticker"),
                    filing_year=sq.get("filing_year"),
                    form_type=sq.get("form_type"),
                )
                chunks = [dict_to_retrieved_chunk(d) for d in retrieved_dicts]
                for chunk in chunks:
                    if chunk.chunk_id not in merged or chunk.score > merged[chunk.chunk_id].score:
                        merged[chunk.chunk_id] = chunk
            retrieved = sorted(merged.values(), key=lambda x: x.score, reverse=True)[: CONFIG["rerank_top_k"]]

        normalized_retrieved = []
        for c in retrieved:
            if isinstance(c, RetrievedChunk):
                normalized_retrieved.append(c)
                continue
            normalized_retrieved.append(
                RetrievedChunk(
                    doc_name=getattr(c, "doc_name", ""),
                    company=getattr(c, "company", ""),
                    ticker=getattr(c, "ticker", ""),
                    form_type=getattr(c, "form_type", ""),
                    filing_year=int(getattr(c, "filing_year", 0) or 0),
                    page_num=int(getattr(c, "page_num", 0) or 0),
                    chunk_id=str(getattr(c, "chunk_id", "")),
                    raw_chunk=getattr(c, "raw_chunk", ""),
                    contextual_chunk=getattr(c, "contextual_chunk", ""),
                    score=float(getattr(c, "score", 0.0) or 0.0),
                    source=getattr(c, "source", "hybrid"),
                )
            )

        state["retrieved_chunks"] = normalized_retrieved
        state["retrieved_doc_names"] = extract_retrieved_doc_names(retrieved)
        state["retrieval_sanity_failed"] = fails_retrieval_sanity_check(
            state["question"],
            sub_queries,
            retrieved,
        )
        
        print(f"    [Retriever] Retrieved {len(retrieved)} chunks | Adjacent expansion: ACTIVE")

        # Event mapping: RetrievedEvent == LangGraph post-retrieval state before context_evaluator.
        return RetrievedEvent(state=state)

In [10]:
# ---------------------------------------------------------------------------
# SEC v3 parity patch for LlamaIndex Workflow
# Adds: drift detection + scrape-on-demand + ingestion retry branch
# ---------------------------------------------------------------------------

import requests
from datetime import datetime, timezone
from collections import defaultdict
from typing import Tuple


TICKER_TO_CIK = {
    "AAPL": "0000320193", "MSFT": "0000789019", "NVDA": "0001045810",
    "JPM": "0000019617", "BAC": "0000070858", "BRK-B": "0001067983",
    "JNJ": "0000200406", "UNH": "0000731766", "XOM": "0000034088",
    "CVX": "0000093410", "WMT": "0000104169", "COST": "0000909832",
}

COMPANY_NAMES_MAP = {
    "AAPL": "Apple Inc.", "MSFT": "Microsoft Corporation", "NVDA": "NVIDIA Corporation",
    "JPM": "JPMorgan Chase & Co.", "BAC": "Bank of America Corporation", "BRK-B": "Berkshire Hathaway Inc.",
    "JNJ": "Johnson & Johnson", "UNH": "UnitedHealth Group Incorporated", "XOM": "Exxon Mobil Corporation",
    "CVX": "Chevron Corporation", "WMT": "Walmart Inc.", "COST": "Costco Wholesale Corporation",
}

# SEC requires a descriptive User-Agent header for EDGAR access.
EDGAR_HEADERS = {"User-Agent": "GenAI-RAG-Project research@example.com"}


class DriftTracker:
    """Tracks insufficient-data misses and schedules scrape targets."""

    TRIGGER_THRESHOLD = 2

    def __init__(self, persist_path: Path):
        self.persist_path = persist_path
        self._counts: Dict[Tuple[str, str, str], int] = defaultdict(int)
        self._already_ingested: set[Tuple[str, str, str]] = set()
        self._load_existing()

    def _load_existing(self) -> None:
        if not self.persist_path.exists():
            return
        with self.persist_path.open("r", encoding="utf-8") as f:
            for line in f:
                try:
                    entry = json.loads(line)
                except Exception:
                    continue
                key = (
                    str(entry.get("ticker") or ""),
                    str(entry.get("filing_year") or ""),
                    str(entry.get("form_type") or "10-K"),
                )
                if entry.get("status") == "ingested":
                    self._already_ingested.add(key)
                elif entry.get("status") == "miss" and all(key):
                    self._counts[key] += 1

    def _append(self, payload: Dict[str, Any]) -> None:
        self.persist_path.parent.mkdir(parents=True, exist_ok=True)
        with self.persist_path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(payload) + "\n")

    def log_miss(self, question: str, ticker: Optional[str], filing_year: Optional[str], form_type: Optional[str]) -> None:
        t = str(ticker or "").upper()
        y = str(filing_year or "")
        f = str(form_type or "10-K").upper()
        key = (t, y, f)
        if not t or not y or key in self._already_ingested:
            return

        self._counts[key] += 1
        self._append(
            {
                "status": "miss",
                "ticker": t,
                "filing_year": y,
                "form_type": f,
                "question": question,
                "timestamp": datetime.now(timezone.utc).isoformat(),
            }
        )
        print(f"  [DriftTracker] Miss #{self._counts[key]} for ({t}, {y}, {f})")

    def get_pending_scrapes(self) -> List[Tuple[str, str, str]]:
        return [
            (t, y, f)
            for (t, y, f), count in self._counts.items()
            if count >= self.TRIGGER_THRESHOLD and (t, y, f) not in self._already_ingested
        ]

    def mark_ingested(self, ticker: str, filing_year: str, form_type: str) -> None:
        key = (str(ticker).upper(), str(filing_year), str(form_type).upper())
        self._already_ingested.add(key)
        self._append(
            {
                "status": "ingested",
                "ticker": key[0],
                "filing_year": key[1],
                "form_type": key[2],
                "timestamp": datetime.now(timezone.utc).isoformat(),
            }
        )


DRIFT_LOG_PATH = resolve_path_from_env(CONFIG["sec_chunks_path"]).parent.parent / "drift_log.jsonl"
drift_tracker = DriftTracker(DRIFT_LOG_PATH)
print(f"DriftTracker ready | persist path: {drift_tracker.persist_path}")


def _strip_html(html: str) -> str:
    text = re.sub(r"<[^>]+>", " ", html)
    for ent, repl in [("&nbsp;", " "), ("&amp;", "&"), ("&lt;", "<"), ("&gt;", ">"), ("&#160;", " ")]:
        text = text.replace(ent, repl)
    return re.sub(r"\s{3,}", "\n\n", text).strip()


def fetch_edgar_filings_list(ticker: str, form_type: str = "10-K") -> List[Dict[str, Any]]:
    cik = TICKER_TO_CIK.get(ticker.upper())
    if not cik:
        raise ValueError(f"Unknown ticker: {ticker}")

    resp = requests.get(
        f"https://data.sec.gov/submissions/CIK{cik}.json",
        headers=EDGAR_HEADERS,
        timeout=30,
    )
    resp.raise_for_status()
    filings = resp.json().get("filings", {}).get("recent", {})

    out: List[Dict[str, Any]] = []
    forms = filings.get("form", [])
    for i, form in enumerate(forms):
        if form != form_type:
            continue
        out.append(
            {
                "accessionNumber": filings.get("accessionNumber", [""])[i],
                "filingDate": filings.get("filingDate", [""])[i],
                "form": form,
                "primaryDocument": (filings.get("primaryDocument") or [None])[i],
            }
        )
    return out


def fetch_filing_text(ticker: str, accession_number: str, primary_doc: Optional[str]) -> str:
    cik = TICKER_TO_CIK[ticker.upper()].lstrip("0")
    acc_no_dashes = accession_number.replace("-", "")
    base = f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc_no_dashes}"

    if primary_doc:
        resp = requests.get(f"{base}/{primary_doc}", headers=EDGAR_HEADERS, timeout=60)
        if resp.ok:
            return _strip_html(resp.text)

    index_url = f"{base}/{accession_number}-index.htm"
    resp = requests.get(index_url, headers=EDGAR_HEADERS, timeout=30)
    links = re.findall(r'href="(/Archives[^"]+\.htm)"', resp.text, re.IGNORECASE)
    if links:
        resp2 = requests.get(f"https://www.sec.gov{links[0]}", headers=EDGAR_HEADERS, timeout=60)
        if resp2.ok:
            return _strip_html(resp2.text)

    raise RuntimeError(f"Could not fetch filing for {ticker} {accession_number}")


def scrape_filing(ticker: str, filing_year: str, form_type: str = "10-K") -> Dict[str, Any]:
    filings = fetch_edgar_filings_list(ticker, form_type)
    match = next((f for f in filings if f["filingDate"].startswith(str(filing_year))), None)
    if match is None:
        match = next((f for f in filings if f["filingDate"].startswith(str(int(filing_year) + 1))), None)
    if match is None:
        raise RuntimeError(f"No {form_type} found for {ticker} FY{filing_year} on EDGAR")

    text = fetch_filing_text(ticker, match["accessionNumber"], match.get("primaryDocument"))
    print(
        f"  [EDGAR] Downloaded {ticker} {form_type} {filing_year} "
        f"({len(text):,} chars, filed {match['filingDate']})"
    )
    return {
        "ticker": ticker.upper(),
        "filing_year": str(filing_year),
        "form_type": form_type.upper(),
        "filing_date": match["filingDate"],
        "accession_number": match["accessionNumber"],
        "text": text,
    }


DRIFT_CHUNK_WORDS = 800
DRIFT_CHUNK_OVERLAP = 100


def chunk_filing(filing: Dict[str, Any]) -> List[Dict[str, Any]]:
    words = filing["text"].split()
    company = COMPANY_NAMES_MAP.get(filing["ticker"], filing["ticker"])

    chunks: List[Dict[str, Any]] = []
    start = 0
    idx = 0
    while start < len(words):
        end = min(start + DRIFT_CHUNK_WORDS, len(words))
        text = " ".join(words[start:end])
        cid = f"{filing['ticker']}_{filing['form_type']}_{filing['filing_year']}_{idx:04d}_drift"
        contextual = (
            f"Company: {company} ({filing['ticker']})\n"
            f"Filing: {filing['form_type']} | Date: {filing['filing_date']} | Year: {filing['filing_year']}\n"
            f"Section: auto-chunked\n"
            f"Content: {text}"
        )
        chunks.append(
            {
                "chunk_id": cid,
                "ticker": filing["ticker"],
                "company_name": company,
                "form_type": filing["form_type"],
                "filing_year": str(filing["filing_year"]),
                "filing_date": filing["filing_date"],
                "accession_number": filing["accession_number"],
                "section_title": "auto-chunked",
                "chunk_index": idx,
                "text": text,
                "contextual_text": contextual,
                "word_count": end - start,
                "source": "drift_ingested",
                "flag_low_value_combined": False,
            }
        )
        start += DRIFT_CHUNK_WORDS - DRIFT_CHUNK_OVERLAP
        idx += 1

    print(f"  [Chunker] {len(chunks)} chunks from {filing['ticker']} {filing['form_type']} {filing['filing_year']}")
    return chunks


def ingest_to_chroma(chunks: List[Dict[str, Any]], corpus_index: CorpusIndex) -> int:
    collection = corpus_index.collection
    ids = [c["chunk_id"] for c in chunks]
    existing = set(collection.get(ids=ids).get("ids", []))
    new_chunks = [c for c in chunks if c["chunk_id"] not in existing]
    if not new_chunks:
        print("  [Ingestor] All chunks already present; skipping.")
        return 0

    texts = [c["contextual_text"] for c in new_chunks]
    embs = embed_model.encode(texts, batch_size=32, show_progress_bar=False).tolist()
    metas = [{k: v for k, v in c.items() if k not in ("text", "contextual_text")} for c in new_chunks]

    collection.upsert(
        ids=[c["chunk_id"] for c in new_chunks],
        embeddings=embs,
        documents=texts,
        metadatas=metas,
    )
    print(f"  [Ingestor] Added {len(new_chunks)} chunks to ChromaDB.")
    return len(new_chunks)


def update_corpus_index(corpus_index: CorpusIndex, new_chunks: List[Dict[str, Any]]) -> None:
    base_len = len(corpus_index.df)
    rows: List[Dict[str, Any]] = []

    for i, c in enumerate(new_chunks):
        doc_name = f"{c['ticker']}_{c['form_type']}_{str(c['filing_date'])[:10]}"
        contextual = c["contextual_text"]
        rows.append(
            {
                "row_idx": base_len + i,
                "chunk_id_str": c["chunk_id"],
                "ticker": c["ticker"],
                "company": c["company_name"],
                "doc_name": doc_name,
                "filing_year": int(c["filing_year"]),
                "form_type": c["form_type"],
                "page_num": int(c["chunk_index"]),
                "raw_chunk": c["text"],
                "contextual_chunk": contextual,
                "bm25_tokens": contextual.lower().split(),
            }
        )

    if rows:
        corpus_index.df = pd.concat([corpus_index.df, pd.DataFrame(rows)], ignore_index=True)
        corpus_index.bm25 = BM25Okapi(corpus_index.df["bm25_tokens"].tolist())

    sec_chunks_out = resolve_path_from_env(CONFIG["sec_chunks_path"])
    sec_chunks_out.parent.mkdir(parents=True, exist_ok=True)
    with sec_chunks_out.open("a", encoding="utf-8") as f:
        for c in new_chunks:
            f.write(json.dumps(c) + "\n")

    print(f"  [BM25] Index rebuilt: {len(corpus_index.df):,} total chunks.")


class DriftDetectEvent(Event):
    state: AgentState


class ScrapeIngestEvent(Event):
    state: AgentState


class JudgeEvent(Event):
    state: AgentState


class SecAgenticRAGWorkflow(Workflow):
    @step
    async def query_planning(self, _ctx: Context, ev: StartEvent) -> PlanEvent:
        question = ev.get("question")
        index = ev.get("index")

        if not question or index is None:
            raise ValueError("Workflow requires question and index")

        state: AgentState = {
            "question": question,
            "index": index,
            "context_retries": 0,
            "repair_used": False,
            "repair_retrieval_count": 0,
            "is_repair_retrieval": False,
            "retrieval_sanity_failed": False,
            "sub_queries": [],
            "needs_decomposition": False,
            "drift_pending_scrapes": [],
            "drift_triggered": False,
            "ingestion_just_ran": False,
            "golden_answer": ev.get("golden_answer"),
            "judge_score": None,
            "judge_claims_covered": None,
            "judge_reason": None,
        }

        result = invoke_structured(
            planner_llm,
            PlannerOutput,
            PLANNER_SYSTEM,
            f"Question:\n{question}",
            PlannerOutput(needs_decomposition=False, sub_queries=[SubQuery(query=question)]),
        )

        sub_queries: List[Dict[str, Any]] = []
        for sq in result.sub_queries:
            sq_dict = sq.model_dump()
            if not sq_dict.get("query"):
                parts = [question]
                if sq_dict.get("ticker"):
                    parts.append(f"ticker {sq_dict['ticker']}")
                if sq_dict.get("filing_year"):
                    parts.append(f"filing year {sq_dict['filing_year']}")
                if sq_dict.get("form_type"):
                    parts.append(f"form {sq_dict['form_type']}")
                sq_dict["query"] = " | ".join(parts)
            sq_dict["query"] = cleanup_planner_query(
                sq_dict.get("query", ""),
                ticker=sq_dict.get("ticker"),
                filing_year=sq_dict.get("filing_year"),
                form_type=sq_dict.get("form_type"),
            )
            sub_queries.append(sq_dict)

        if not sub_queries:
            sub_queries = [
                {
                    "query": cleanup_planner_query(question),
                    "ticker": None,
                    "filing_year": None,
                    "form_type": None,
                }
            ]

        state["sub_queries"] = sub_queries
        state["rewritten_query"] = sub_queries[0]["query"]
        state["needs_decomposition"] = result.needs_decomposition

        print(
            f"  [Planner] {'decomposed' if state['needs_decomposition'] else 'single'} | "
            f"{len(sub_queries)} sub-quer{'ies' if len(sub_queries) > 1 else 'y'}"
        )

        if state["needs_decomposition"]:
            for sq in sub_queries:
                print(
                    f"    -> {sq['query']}  "
                    f"(ticker={sq.get('ticker')}, year={sq.get('filing_year')}, form={sq.get('form_type')})"
                )

        return PlanEvent(state=state)

    @step
    async def hybrid_retriever(
        self,
        _ctx: Context,
        ev: PlanEvent | RetrieveEvent | RepairRetrieveEvent,
    ) -> RetrievedEvent:
        state = ev.state
        sub_queries = state.get("sub_queries", [])

        is_repair_retrieval = isinstance(ev, RepairRetrieveEvent)
        if is_repair_retrieval:
            state["is_repair_retrieval"] = True

        if len(sub_queries) <= 1:
            q = state.get("repair_query") if is_repair_retrieval else state.get("rewritten_query", state["question"])
            q = q or state.get("rewritten_query", state["question"])
            sq = sub_queries[0] if sub_queries else {}
            repair_ticker = sq.get("ticker") if is_repair_retrieval else sq.get("ticker")
            repair_filing_year = None if is_repair_retrieval else sq.get("filing_year")
            repair_form_type = None if is_repair_retrieval else sq.get("form_type")
            bm25_top_k = int(CONFIG["bm25_top_k"] * (2 if is_repair_retrieval else 1))
            dense_top_k = int(CONFIG["dense_top_k"] * (2 if is_repair_retrieval else 1))
            rerank_top_k = int(CONFIG["rerank_top_k"] * (2 if is_repair_retrieval else 1))
            active_index = state.get("index") or globals().get("global_index")
            if active_index is None:
                raise KeyError("index")
            retrieved = active_index.hybrid_search(
                query=q,
                # shared retriever uses models from its own index state,
                bm25_top_k=bm25_top_k,
                dense_top_k=dense_top_k,
                rerank_top_k=rerank_top_k,
                ticker=repair_ticker,
                filing_year=repair_filing_year,
                form_type=repair_form_type,
            )
        else:
            per_k = CONFIG["decomposition_top_k_per_subquery"]
            merged: Dict[str, RetrievedChunk] = {}
            for sq in sub_queries:
                active_index = state.get("index") or globals().get("global_index")
                if active_index is None:
                    raise KeyError("index")
                chunks = active_index.hybrid_search(
                    query=sq["query"],
                    # shared retriever uses models from its own index state,
                    bm25_top_k=CONFIG["bm25_top_k"],
                    dense_top_k=CONFIG["dense_top_k"],
                    rerank_top_k=per_k,
                    ticker=sq.get("ticker"),
                    filing_year=sq.get("filing_year"),
                    form_type=sq.get("form_type"),
                )
                for chunk in chunks:
                    if chunk.chunk_id not in merged or chunk.score > merged[chunk.chunk_id].score:
                        merged[chunk.chunk_id] = chunk

            retrieved = sorted(merged.values(), key=lambda x: x.score, reverse=True)[: CONFIG["rerank_top_k"]]

        state["retrieved_chunks"] = retrieved
        state["retrieved_doc_names"] = extract_retrieved_doc_names(retrieved)
        state["retrieval_sanity_failed"] = fails_retrieval_sanity_check(
            state["question"],
            sub_queries,
            retrieved,
        )

        return RetrievedEvent(state=state)

    @step
    async def context_evaluator(self, _ctx: Context, ev: RetrievedEvent) -> GenerateEvent | RetrieveEvent:
        state = ev.state
        retrieval_signature = (
            tuple(state.get("retrieved_doc_names", [])[:10]),
            len(state.get("retrieved_chunks", []) or []),
        )

        if state.get("retrieval_sanity_failed", False):
            relevant = False
        else:
            context = format_retrieved_context(
                state.get("retrieved_chunks", []),
                max_chunks=CONFIG["control_max_context_chunks"],
                max_chars=CONFIG["control_max_context_chars"],
            )
            judged = invoke_structured(
                context_eval_llm,
                ContextEvalOutput,
                CONTEXT_EVAL_SYSTEM,
                f"Question:\n{state['question']}\n\nContext:\n{context}",
                ContextEvalOutput(relevant=True, reason="fallback"),
            )
            relevant = judged.relevant

        state["context_eval_relevant"] = relevant

        if relevant:
            state["last_retrieval_signature"] = retrieval_signature
            return GenerateEvent(state=state)

        retries = state.get("context_retries", 0)
        prev_signature = state.get("last_retrieval_signature")
        identical_retry = bool(
            CONFIG.get("avoid_identical_retrieval_retries", True)
            and prev_signature is not None
            and prev_signature == retrieval_signature
        )
        state["last_retrieval_signature"] = retrieval_signature
        if retries < CONFIG["max_context_retries"] and not identical_retry:
            state["context_retries"] = retries + 1
            return RetrieveEvent(state=state)
        if identical_retry:
            state["context_retry_skipped_reason"] = "identical_retrieval_signature"

        return GenerateEvent(state=state)

    @step
    async def generator(self, _ctx: Context, ev: GenerateEvent) -> CritiqueEvent:
        state = ev.state
        context = format_retrieved_context(
            state.get("retrieved_chunks", []),
            max_chunks=CONFIG["generator_max_context_chunks"],
            max_chars=CONFIG["generator_max_context_chars"],
        )

        answer = invoke_text(
            generator_llm,
            GENERATOR_SYSTEM,
            f"Question:\n{state['question']}\n\nRetrieved context:\n{context}",
            fallback_text="",
        ).strip()

        state["generated_answer"] = answer
        state["final_answer"] = answer
        return CritiqueEvent(state=state)

    @step
    async def critic(self, _ctx: Context, ev: CritiqueEvent) -> JudgeEvent | RepairEvent | DriftDetectEvent:
        state = ev.state

        if state.get("is_repair_retrieval", False):
            return JudgeEvent(state=state)

        context = format_retrieved_context(
            state.get("retrieved_chunks", []),
            max_chunks=CONFIG["control_max_context_chunks"],
            max_chars=CONFIG["control_max_context_chars"],
        )

        result = invoke_structured(
            critic_llm,
            CriticOutput,
            CRITIC_SYSTEM,
            (
                f"Question:\n{state['question']}\n\n"
                f"Context:\n{context}\n\n"
                f"Answer:\n{state.get('generated_answer', '')}"
            ),
            CriticOutput(decision="accept", reason="fallback"),
        )

        state["critic_decision"] = result.decision
        state["critic_reason"] = result.reason

        if result.decision == "repair":
            return RepairEvent(state=state)

        if result.decision == "insufficient_data":
            state["final_answer"] = (
                "Insufficient data: The required information could not be found in the "
                f"retrieved SEC filings. ({result.reason})"
            )
            if CONFIG.get("enable_drift_and_scrape", False):
                return DriftDetectEvent(state=state)
            return JudgeEvent(state=state)

        return JudgeEvent(state=state)

    @step
    async def repair(self, _ctx: Context, ev: RepairEvent) -> JudgeEvent | RepairRetrieveEvent:
        state = ev.state
        context = format_retrieved_context(
            state.get("retrieved_chunks", []),
            max_chunks=CONFIG["control_max_context_chunks"],
            max_chars=CONFIG["control_max_context_chars"],
        )

        critique_reason = str(state.get("critic_reason", "Needs repair"))
        result = invoke_structured(
            repair_llm,
            RepairOutput,
            REPAIR_SYSTEM,
            (
                f"Question:\n{state['question']}\n\n"
                f"Context:\n{context}\n\n"
                f"Answer:\n{state.get('generated_answer', '')}\n\n"
                f"Critique:\n{critique_reason}"
            ),
            RepairOutput(
                decision="accept",
                revised_answer=state.get("generated_answer", ""),
                needs_new_retrieval=False,
                reason="fallback",
            ),
        )

        critique_lower = critique_reason.lower()
        heuristic_new_retrieval = any(
            token in critique_lower
            for token in [
                "omit", "omits", "missing", "missed", "incomplete", "not enough", "additional",
                "other", "broader", "coverage", "wrong source", "wrong document", "wrong filing",
                "unsupported", "ungrounded", "hallucinat", "incorrect date", "incorrect form",
                "not present in the context", "not in the context", "fails grounding", "source document",
                "filing date", "10-k", "10-q",
            ]
        )
        state["repair_used"] = True
        state["repair_decision"] = result.decision
        state["repair_reason"] = result.reason
        state["needs_new_retrieval"] = bool(result.needs_new_retrieval or heuristic_new_retrieval)
        state["final_answer"] = (result.revised_answer or "").strip()
        state["repair_query"] = (
            f"{state['question']} Recover missing grounded evidence only. Critique: {critique_reason}"
            if state.get("needs_new_retrieval", False)
            else state.get("question")
        )

        if state.get("needs_new_retrieval", False):
            count = state.get("repair_retrieval_count", 0) + 1
            state["repair_retrieval_count"] = count
            if count <= CONFIG["max_repair_retrievals"]:
                state["is_repair_retrieval"] = True
                return RepairRetrieveEvent(state=state)

        state["is_repair_retrieval"] = False
        return JudgeEvent(state=state)

    @step
    async def drift_detector(self, _ctx: Context, ev: DriftDetectEvent) -> JudgeEvent | ScrapeIngestEvent:
        state = ev.state
        sub_queries = state.get("sub_queries", [])

        if not CONFIG.get("enable_drift_and_scrape", False):
            return JudgeEvent(state=state)

        ticker = None
        filing_year = None
        form_type = "10-K"
        if sub_queries:
            sq = sub_queries[0]
            ticker = sq.get("ticker")
            filing_year = str(sq.get("filing_year")) if sq.get("filing_year") else None
            form_type = str(sq.get("form_type") or "10-K")

        drift_tracker.log_miss(
            question=state["question"],
            ticker=ticker,
            filing_year=filing_year,
            form_type=form_type,
        )

        pending = drift_tracker.get_pending_scrapes()
        state["drift_pending_scrapes"] = pending
        state["drift_triggered"] = len(pending) > 0
        state["ingestion_just_ran"] = False

        print(f"  [DriftDetector] Pending scrapes: {pending}")

        if state["drift_triggered"] and pending:
            return ScrapeIngestEvent(state=state)

        return JudgeEvent(state=state)

    @step
    async def scrape_and_ingest(self, _ctx: Context, ev: ScrapeIngestEvent) -> JudgeEvent | RetrieveEvent:
        state = ev.state
        corpus_index = state["index"]
        ingested_any = False

        for ticker, filing_year, form_type in state.get("drift_pending_scrapes", []):
            try:
                filing = scrape_filing(ticker, str(filing_year), form_type)
                chunks = chunk_filing(filing)
                added = ingest_to_chroma(chunks, corpus_index)
                if added > 0:
                    update_corpus_index(corpus_index, chunks)
                    ingested_any = True
                drift_tracker.mark_ingested(ticker, str(filing_year), form_type)
            except Exception as e:
                print(f"  [ScrapeIngest] Failed for {ticker} {filing_year}: {e}")

        state["ingestion_just_ran"] = ingested_any
        state["context_retries"] = 0
        state["critic_decision"] = None
        state["is_repair_retrieval"] = False

        if ingested_any:
            return RetrieveEvent(state=state)

        return JudgeEvent(state=state)

    @step
    async def judge_final_answer(self, _ctx: Context, ev: JudgeEvent) -> StopEvent:
        state = ev.state
        reference = str(state.get("golden_answer") or "").strip()
        candidate = str(state.get("final_answer") or "").strip()

        if reference:
            score, claims, reason = llm_judge_correctness(
                state["question"],
                reference,
                candidate,
            )
            state["judge_score"] = score
            state["judge_claims_covered"] = claims
            state["judge_reason"] = reason
        else:
            state["judge_score"] = None
            state["judge_claims_covered"] = None
            state["judge_reason"] = "Skipped: no golden_answer supplied."

        return StopEvent(result=state)


print("SecAgenticRAGWorkflow updated (judge integrated; drift toggle enabled).")

DriftTracker ready | persist path: ..\sec_rag_team_share\sec_data\drift_log.jsonl
SecAgenticRAGWorkflow updated (judge integrated; drift toggle enabled).


In [11]:
# Compatibility shims used by downstream parity-alignment cells.
if 'RateLimiter' not in globals():
    class RateLimiter:
        def __init__(self, max_rpm: int):
            self.max_rpm = max_rpm
            self.window = 60.0
            self._timestamps = deque()
            self._lock = threading.Lock()

        def wait(self):
            with self._lock:
                now = time.time()
                while self._timestamps and now - self._timestamps[0] >= self.window:
                    self._timestamps.popleft()
                if len(self._timestamps) >= self.max_rpm:
                    sleep_for = self.window - (now - self._timestamps[0])
                    if sleep_for > 0:
                        time.sleep(sleep_for)
                self._timestamps.append(time.time())

if 'resolve_role_temperature' not in globals():
    def resolve_role_temperature(role: str, task_name: Optional[str] = None) -> float:
        if task_name:
            tk = f"temperature_{task_name}"
            if tk in CONFIG:
                return float(CONFIG[tk])
        rk = f"temperature_{role}"
        if rk in CONFIG:
            return float(CONFIG[rk])
        return float(CONFIG.get('temperature_generator', 0.2))

print('Compatibility shims ready (RateLimiter, resolve_role_temperature).')

Compatibility shims ready (RateLimiter, resolve_role_temperature).


In [12]:
# ---------------------------------------------------------------------------
# Parity alignment patch: bring LlamaIndex workflow behavior closer to
# LangGraph SEC v3 while preserving path portability logic.
# ---------------------------------------------------------------------------

# 1) Configuration alignment (do not alter path portability keys)
CONFIG["execution_profile"] = "dev"
CONFIG["use_pilot"] = False
CONFIG["pilot_n_questions"] = 5
CONFIG["full_n_questions"] = 80
CONFIG["run_smoke_test"] = False
CONFIG.setdefault("llm_request_timeout_sec", 90)
CONFIG.setdefault("workflow_timeout_sec", 240)
CONFIG.setdefault("max_context_retries", 2)
CONFIG.setdefault("max_repair_retrievals", 1)
CONFIG.setdefault("avoid_identical_retrieval_retries", True)
CONFIG.setdefault("drift_only_on_empty_context", True)

# Gemini override for this notebook: use gemini-2.5-flash for every role/profile.
if CONFIG.get("provider") == "gemini":
    CONFIG["gemini_dev_generator_model"] = "gemini-2.5-flash"
    CONFIG["gemini_dev_agent_model"] = "gemini-2.5-flash"
    CONFIG["gemini_dev_judge_model"] = "gemini-2.5-flash"
    CONFIG["gemini_final_generator_model"] = "gemini-2.5-flash"
    CONFIG["gemini_final_agent_model"] = "gemini-2.5-flash"
    CONFIG["gemini_final_judge_model"] = "gemini-2.5-flash"
CONFIG["use_llm_judge"] = False
CONFIG["use_few_shot_examples"] = True
CONFIG["pilot_judge_sample_n"] = 1
CONFIG["full_judge_sample_n"] = 2
CONFIG["judge_max_context_chunks"] = 3
CONFIG["judge_max_context_chars"] = 4000

# Keep existing drift toggle semantics in drift_detector,
# but update critic routing to always pass through drift_detector.
CONFIG["enable_drift_and_scrape"] = bool(
    CONFIG.get("enable_drift_and_scrape", CONFIG.get("ENABLE_DRIFT_AND_SCRAPE", False))
)

# LangGraph-style provider aware RPM.
CONFIG["max_rpm"] = 28 if CONFIG["provider"] == "groq" else 10
CONFIG["judge_sample_n"] = (
    CONFIG["pilot_judge_sample_n"] if CONFIG["use_pilot"] else CONFIG["full_judge_sample_n"]
) if CONFIG["use_llm_judge"] else 0

# LangGraph-style fallback model lists.
if CONFIG["provider"] == "groq":
    CONFIG.setdefault("groq_fallback_agent_models", ["llama-3.1-8b-instant", "qwen/qwen3-32b"])
    CONFIG.setdefault("groq_fallback_generator_models", ["llama-3.1-8b-instant", "qwen/qwen3-32b"])
    CONFIG.setdefault("groq_fallback_judge_models", ["llama-3.1-8b-instant", "qwen/qwen3-32b"])
if CONFIG["provider"] == "gemini":
    CONFIG.setdefault("gemini_fallback_agent_models", [])
    CONFIG.setdefault("gemini_fallback_generator_models", [])
    CONFIG.setdefault("gemini_fallback_judge_models", [])

# Rebuild limiter after max_rpm update.
rate_limiter = RateLimiter(CONFIG["max_rpm"])


def make_llm_from_model(model_name: str, temperature: float):
    if CONFIG["provider"] == "groq":
        api_key = os.getenv("GROQ_API_KEY", "")
        if not api_key:
            raise ValueError("GROQ_API_KEY is missing in .env")
        return LIGroq(model=model_name, api_key=api_key, temperature=temperature)

    if CONFIG["provider"] == "gemini":
        api_key = os.getenv("GOOGLE_API_KEY", "").strip() or os.getenv("GEMINI_API_KEY", "").strip()
        if not api_key:
            raise ValueError("Missing Gemini API key. Set GOOGLE_API_KEY or GEMINI_API_KEY in the environment/.env.")
        return LIGoogleGenAI(model=model_name, api_key=api_key, temperature=temperature)

    raise ValueError(f"Unsupported provider: {CONFIG['provider']}")


def make_llm(role: str, task_name: Optional[str] = None):
    model_name = resolve_model_name(role)
    temp = resolve_role_temperature(role, task_name)
    return make_llm_from_model(model_name, temp)


def resolve_fallback_model_names(role: str) -> List[str]:
    provider = CONFIG["provider"]
    primary = resolve_model_name(role)
    fallbacks = CONFIG.get(f"{provider}_fallback_{role}_models", [])
    ordered = [primary] + list(fallbacks)
    return list(dict.fromkeys([m for m in ordered if m]))


def get_llm_candidates(role: str, task_name: Optional[str] = None):
    temperature = resolve_role_temperature(role, task_name)
    return [
        (model_name, make_llm_from_model(model_name, temperature))
        for model_name in resolve_fallback_model_names(role)
    ]


_preferred_models_by_task: Dict[str, str] = {}


def order_model_candidates(role: str, task_name: Optional[str] = None):
    candidates = get_llm_candidates(role, task_name)
    preference_key = task_name or role
    preferred = _preferred_models_by_task.get(preference_key)
    if not preferred:
        return candidates
    names = [name for name, _ in candidates]
    if preferred not in names:
        return candidates
    return sorted(candidates, key=lambda item: item[0] != preferred)


def is_rate_limit_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    return "rate limit" in msg or "rate_limit" in msg or "429" in msg


def _infer_role_task_from_llm(llm_obj) -> Optional[tuple[str, str]]:
    if llm_obj is planner_llm:
        return ("agent", "planner")
    if llm_obj is context_eval_llm:
        return ("agent", "context_eval")
    if llm_obj is generator_llm:
        return ("generator", "generator")
    if llm_obj is critic_llm:
        return ("agent", "critic")
    if llm_obj is repair_llm:
        return ("agent", "repair")
    if llm_obj is judge_llm:
        return ("judge", "judge")
    return None


def _question_tokens(text: str) -> set[str]:
    return {tok for tok in re.findall(r"[a-z0-9]+", str(text).lower()) if len(tok) > 2}


def select_few_shot_examples(question: str, max_examples: int = 2) -> str:
    if not CONFIG.get("use_few_shot_examples", False):
        return ""
    if "train_df" not in globals():
        return ""
    if train_df is None or len(train_df) == 0:
        return ""

    q_tokens = _question_tokens(question)
    scored: List[tuple[int, str, str]] = []
    for _, row in train_df.iterrows():
        ex_q = str(row.get("question", "")).strip()
        ex_a = str(row.get("reference_answer", "")).strip()
        if not ex_q or not ex_a:
            continue
        overlap = len(q_tokens & _question_tokens(ex_q))
        scored.append((overlap, ex_q, ex_a))

    if not scored:
        return ""

    scored.sort(key=lambda x: x[0], reverse=True)
    top = scored[:max_examples]
    lines: List[str] = []
    for i, (_, ex_q, ex_a) in enumerate(top, start=1):
        lines.append(f"Example {i} Question: {ex_q}")
        lines.append(f"Example {i} Answer: {ex_a}")
    return "\n".join(lines)


def _extract_question_from_generator_user_block(user: str) -> str:
    m = re.search(r"Question:\n(.*?)\n\nRetrieved context:", user, flags=re.DOTALL)
    if not m:
        return ""
    return m.group(1).strip()


def invoke_structured(
    llm_or_role,
    schema,
    system: str,
    user: str,
    fallback,
    task_name: Optional[str] = None,
):
    prompt = f"System:\n{system}\n\nUser:\n{user}"

    # Preferred path: role-based fallback strategy.
    if isinstance(llm_or_role, str):
        role = llm_or_role
        candidates = order_model_candidates(role, task_name)
        preference_key = task_name or role
        max_attempts = int(CONFIG["llm_max_retries"])

        for model_idx, (model_name, llm) in enumerate(candidates):
            for attempt in range(max_attempts):
                try:
                    raw = llm_complete(llm, prompt)
                    data = json.loads(_strip_code_fence(raw))
                    parsed = schema.model_validate(data)
                    _preferred_models_by_task[preference_key] = model_name
                    return parsed
                except Exception as e:
                    print(
                        f"  [WARN] {schema.__name__} attempt {attempt + 1}/{max_attempts} "
                        f"on {model_name} failed: {e}"
                    )
                    should_switch = is_rate_limit_error(e) and model_idx < len(candidates) - 1
                    if should_switch:
                        break
                    if attempt < max_attempts - 1:
                        time.sleep(CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt))
        return fallback

    # Backward compatibility path: infer role/task from llm object if possible.
    inferred = _infer_role_task_from_llm(llm_or_role)
    if inferred is not None:
        role, inferred_task = inferred
        return invoke_structured(
            role,
            schema,
            system,
            user,
            fallback,
            task_name=task_name or inferred_task,
        )

    # Last resort legacy behavior for unknown llm objects.
    for attempt in range(int(CONFIG["llm_max_retries"])):
        try:
            raw = llm_complete(llm_or_role, prompt)
            data = json.loads(_strip_code_fence(raw))
            return schema.model_validate(data)
        except Exception:
            if attempt == int(CONFIG["llm_max_retries"]) - 1:
                return fallback
            time.sleep(CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt))
    return fallback


def invoke_text(
    llm_or_role,
    system: str,
    user: str,
    fallback_text: str = "",
    task_name: Optional[str] = None,
) -> str:
    # Preferred path: role-based fallback strategy.
    if isinstance(llm_or_role, str):
        role = llm_or_role
        candidates = order_model_candidates(role, task_name)
        preference_key = task_name or role
        max_attempts = int(CONFIG["llm_max_retries"])

        effective_user = user
        if (task_name or "") == "generator":
            question = _extract_question_from_generator_user_block(user)
            few_shot = select_few_shot_examples(question)
            if few_shot:
                effective_user = f"Few-shot examples:\n{few_shot}\n\n{user}"

        prompt = f"System:\n{system}\n\nUser:\n{effective_user}"

        for model_idx, (model_name, llm) in enumerate(candidates):
            for attempt in range(max_attempts):
                try:
                    result = llm_complete(llm, prompt)
                    _preferred_models_by_task[preference_key] = model_name
                    return result
                except Exception as e:
                    print(
                        f"  [WARN] text invoke attempt {attempt + 1}/{max_attempts} "
                        f"on {model_name} failed: {e}"
                    )
                    should_switch = is_rate_limit_error(e) and model_idx < len(candidates) - 1
                    if should_switch:
                        break
                    if attempt < max_attempts - 1:
                        time.sleep(CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt))
        return fallback_text

    inferred = _infer_role_task_from_llm(llm_or_role)
    if inferred is not None:
        role, inferred_task = inferred
        return invoke_text(
            role,
            system,
            user,
            fallback_text=fallback_text,
            task_name=task_name or inferred_task,
        )

    prompt = f"System:\n{system}\n\nUser:\n{user}"
    try:
        return llm_complete(llm_or_role, prompt)
    except Exception:
        return fallback_text


def print_active_model_config() -> None:
    provider = CONFIG.get("provider", "unknown")
    profile = CONFIG.get("execution_profile", "unknown")
    role_models = {
        "agent": resolve_model_name("agent"),
        "generator": resolve_model_name("generator"),
        "judge": resolve_model_name("judge"),
    }
    print("=" * 60)
    print(" Active LLM Configuration")
    print("=" * 60)
    print(f"Provider           : {provider}")
    print(f"Execution profile  : {profile}")
    print(f"Agent model        : {role_models['agent']}")
    print(f"Generator model    : {role_models['generator']}")
    print(f"Judge model        : {role_models['judge']}")
    print("Task mapping:")
    print(f"  planner          -> {role_models['agent']}")
    print(f"  context_eval     -> {role_models['agent']}")
    print(f"  critic           -> {role_models['agent']}")
    print(f"  repair           -> {role_models['agent']}")
    print(f"  generator        -> {role_models['generator']}")
    print(f"  judge            -> {role_models['judge']}")
    print("=" * 60)


# Recreate role llm handles after execution profile update.
planner_llm = make_llm("agent", task_name="planner")
context_eval_llm = make_llm("agent", task_name="context_eval")
generator_llm = make_llm("generator", task_name="generator")
critic_llm = make_llm("agent", task_name="critic")
repair_llm = make_llm("agent", task_name="repair")
judge_llm = make_llm("judge", task_name="judge")
print_active_model_config()

JUDGE_CORRECTNESS_WITH_CONTEXT_SYSTEM = (
    "You are a strict evaluator. Score correctness between 0.0 and 1.0 and claims_covered between 0.0 and 1.0. "
    "Use the reference answer as the source of truth and use provided retrieved context for grounding checks. "
    "Return JSON with keys: score, claims_covered, reason."
)


# 2) Workflow behavior alignment
#    - critic routes insufficient_data to drift detector (like LangGraph graph path)
#    - judge uses explicit judge context caps
_SecAgenticRAGWorkflowBase = SecAgenticRAGWorkflow


class SecAgenticRAGWorkflow(_SecAgenticRAGWorkflowBase):
    @step
    async def critic(self, _ctx: Context, ev: CritiqueEvent) -> JudgeEvent | RepairEvent | DriftDetectEvent:
        state = ev.state

        if state.get("is_repair_retrieval", False):
            return JudgeEvent(state=state)

        context = format_retrieved_context(
            state.get("retrieved_chunks", []),
            max_chunks=CONFIG["control_max_context_chunks"],
            max_chars=CONFIG["control_max_context_chars"],
        )

        result = invoke_structured(
            "agent",
            CriticOutput,
            CRITIC_SYSTEM,
            (
                f"Question:\n{state['question']}\n\n"
                f"Context:\n{context}\n\n"
                f"Answer:\n{state.get('generated_answer', '')}"
            ),
            CriticOutput(decision="accept", reason="fallback"),
            task_name="critic",
        )

        state["critic_decision"] = result.decision
        state["critic_reason"] = result.reason

        if result.decision == "repair":
            return RepairEvent(state=state)

        if result.decision == "insufficient_data":
            state["final_answer"] = (
                "Insufficient data: The required information could not be found in the "
                f"retrieved SEC filings. ({result.reason})"
            )
            has_retrieved_context = bool(state.get("retrieved_chunks"))
            should_drift = bool(CONFIG.get("enable_drift_and_scrape", False))
            if CONFIG.get("drift_only_on_empty_context", True):
                should_drift = should_drift and (not has_retrieved_context or state.get("retrieval_sanity_failed", False))
            if should_drift:
                return DriftDetectEvent(state=state)
            return JudgeEvent(state=state)

        return JudgeEvent(state=state)

    @step
    async def judge_final_answer(self, _ctx: Context, ev: JudgeEvent) -> StopEvent:
        state = ev.state
        reference = str(state.get("golden_answer") or "").strip()
        candidate = str(state.get("final_answer") or "").strip()

        judge_context = format_retrieved_context(
            state.get("retrieved_chunks", []),
            max_chunks=int(CONFIG.get("judge_max_context_chunks", 3)),
            max_chars=int(CONFIG.get("judge_max_context_chars", 4000)),
        )

        if reference:
            judged = invoke_structured(
                "judge",
                JudgeOutput,
                JUDGE_CORRECTNESS_WITH_CONTEXT_SYSTEM,
                (
                    f"Question:\n{state['question']}\n\n"
                    f"Reference answer:\n{reference}\n\n"
                    f"Retrieved context:\n{judge_context}\n\n"
                    f"Candidate answer:\n{candidate}"
                ),
                JudgeOutput(score=0.0, claims_covered=0.0, reason="Judge fallback"),
                task_name="judge",
            )
            state["judge_score"] = max(0.0, min(1.0, float(judged.score)))
            state["judge_claims_covered"] = max(0.0, min(1.0, float(judged.claims_covered)))
            state["judge_reason"] = judged.reason
        else:
            state["judge_score"] = None
            state["judge_claims_covered"] = None
            state["judge_reason"] = "Skipped: no golden_answer supplied."

        return StopEvent(result=state)


print("Parity alignment patch applied (config + fallback + few-shot + drift/judge behavior).")

 Active LLM Configuration
Provider           : gemini
Execution profile  : dev
Agent model        : gemini-2.5-flash
Generator model    : gemini-2.5-flash
Judge model        : gemini-2.5-flash
Task mapping:
  planner          -> gemini-2.5-flash
  context_eval     -> gemini-2.5-flash
  critic           -> gemini-2.5-flash
  repair           -> gemini-2.5-flash
  generator        -> gemini-2.5-flash
  judge            -> gemini-2.5-flash
Parity alignment patch applied (config + fallback + few-shot + drift/judge behavior).


In [13]:
# ---------------------------------------------------------------------------
# JSON parsing hardening + judge prompt refinement
# ---------------------------------------------------------------------------


def _strip_code_fence(text: str) -> str:
    """Extract the first JSON object-like block from noisy model output."""
    if text is None:
        return ""
    s = str(text).strip()
    match = re.search(r"\{[\s\S]*\}", s)
    if match:
        return match.group(0).strip()
    return s


def invoke_structured(
    llm_or_role,
    schema,
    system: str,
    user: str,
    fallback,
    task_name: Optional[str] = None,
):
    prompt = f"System:\n{system}\n\nUser:\n{user}"

    # Preferred path: role-based fallback strategy.
    if isinstance(llm_or_role, str):
        role = llm_or_role
        candidates = order_model_candidates(role, task_name)
        preference_key = task_name or role
        max_attempts = int(CONFIG["llm_max_retries"])

        for model_idx, (model_name, llm) in enumerate(candidates):
            for attempt in range(max_attempts):
                try:
                    raw = llm_complete(llm, prompt)
                    cleaned = _strip_code_fence(raw)
                    try:
                        data = json.loads(cleaned)
                    except json.JSONDecodeError as jde:
                        print(
                            f"  [WARN] {schema.__name__} JSON decode failed on {model_name} "
                            f"attempt {attempt + 1}/{max_attempts}: {jde}"
                        )
                        print("  [RAW LLM OUTPUT START]")
                        print(raw)
                        print("  [RAW LLM OUTPUT END]")
                        raise
                    parsed = schema.model_validate(data)
                    _preferred_models_by_task[preference_key] = model_name
                    return parsed
                except Exception as e:
                    print(
                        f"  [WARN] {schema.__name__} attempt {attempt + 1}/{max_attempts} "
                        f"on {model_name} failed: {e}"
                    )
                    should_switch = is_rate_limit_error(e) and model_idx < len(candidates) - 1
                    if should_switch:
                        break
                    if attempt < max_attempts - 1:
                        time.sleep(CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt))
        return fallback

    # Backward compatibility path: infer role/task from llm object if possible.
    inferred = _infer_role_task_from_llm(llm_or_role)
    if inferred is not None:
        role, inferred_task = inferred
        return invoke_structured(
            role,
            schema,
            system,
            user,
            fallback,
            task_name=task_name or inferred_task,
        )

    # Last resort legacy behavior for unknown llm objects.
    for attempt in range(int(CONFIG["llm_max_retries"])):
        try:
            raw = llm_complete(llm_or_role, prompt)
            cleaned = _strip_code_fence(raw)
            try:
                data = json.loads(cleaned)
            except json.JSONDecodeError as jde:
                print(
                    f"  [WARN] {schema.__name__} legacy JSON decode failed "
                    f"attempt {attempt + 1}/{int(CONFIG['llm_max_retries'])}: {jde}"
                )
                print("  [RAW LLM OUTPUT START]")
                print(raw)
                print("  [RAW LLM OUTPUT END]")
                raise
            return schema.model_validate(data)
        except Exception:
            if attempt == int(CONFIG["llm_max_retries"]) - 1:
                return fallback
            time.sleep(CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt))
    return fallback


JUDGE_CORRECTNESS_WITH_CONTEXT_SYSTEM = """
<role>
You are a strict financial QA judge for SEC filing question answering.
</role>

<task>
Evaluate candidate_answer against reference_answer, using retrieved_context only to check grounding.
Return score and claims_covered in [0.0, 1.0].
</task>

<rubric>
- score: overall factual correctness vs reference (numbers, entities, period, directionality).
- claims_covered: fraction of key reference claims covered by candidate.
- reason: concise rationale (1-3 sentences).
</rubric>

<output_format>
Return exactly one JSON object with keys: score, claims_covered, reason.
</output_format>

<hard_constraint>
Do not include any introductory text or summary. Start your response with { and end with }.
</hard_constraint>
""".strip()


def _judge_few_shot_examples() -> str:
    return (
        "Example A (Perfect)\n"
        "Reference: Apple gross margin rose from 44.1% in FY2023 to 46.6% in FY2024.\n"
        "Candidate: Apple's gross margin increased from 44.1% in FY2023 to 46.6% in FY2024.\n"
        "Expected JSON: {\"score\": 1.0, \"claims_covered\": 1.0, \"reason\": \"Matches both values and trend exactly.\"}\n\n"
        "Example B (Poor)\n"
        "Reference: NVIDIA data center revenue increased year over year.\n"
        "Candidate: NVIDIA data center revenue declined sharply year over year.\n"
        "Expected JSON: {\"score\": 0.0, \"claims_covered\": 0.0, \"reason\": \"Contradicts the reference trend.\"}"
    )


_SecAgenticRAGWorkflowJudgeBase = SecAgenticRAGWorkflow


class SecAgenticRAGWorkflow(_SecAgenticRAGWorkflowJudgeBase):
    @step
    async def judge_final_answer(self, _ctx: Context, ev: JudgeEvent) -> StopEvent:
        state = ev.state
        reference = str(state.get("golden_answer") or "").strip()
        candidate = str(state.get("final_answer") or "").strip()

        judge_context = format_retrieved_context(
            state.get("retrieved_chunks", []),
            max_chunks=int(CONFIG.get("judge_max_context_chunks", 3)),
            max_chars=int(CONFIG.get("judge_max_context_chars", 4000)),
        )

        if reference:
            few_shot = _judge_few_shot_examples() if CONFIG.get("use_few_shot_examples", False) else ""
            judged = invoke_structured(
                "judge",
                JudgeOutput,
                JUDGE_CORRECTNESS_WITH_CONTEXT_SYSTEM,
                (
                    (f"Scoring examples:\n{few_shot}\n\n" if few_shot else "")
                    + f"Question:\n{state['question']}\n\n"
                    + f"Reference answer:\n{reference}\n\n"
                    + f"Retrieved context:\n{judge_context}\n\n"
                    + f"Candidate answer:\n{candidate}"
                ),
                JudgeOutput(score=0.0, claims_covered=0.0, reason="Judge fallback"),
                task_name="judge",
            )
            state["judge_score"] = max(0.0, min(1.0, float(judged.score)))
            state["judge_claims_covered"] = max(0.0, min(1.0, float(judged.claims_covered)))
            state["judge_reason"] = judged.reason
        else:
            state["judge_score"] = None
            state["judge_claims_covered"] = None
            state["judge_reason"] = "Skipped: no golden_answer supplied."

        return StopEvent(result=state)


print("JSON parsing and judge prompt hardening patch applied.")

JSON parsing and judge prompt hardening patch applied.


In [14]:
def _looks_enumeration_question(question: str) -> bool:
    q = str(question or "").lower()
    triggers = [
        "what are", "which are", "list ", "risk factors", "latest risk factors",
        "primary", "main", "key", "drivers", "causes", "segments",
    ]
    return any(t in q for t in triggers)


def _enumerated_item_count(text: str) -> int:
    s = str(text or "").strip()
    if not s:
        return 0
    lines = [line.strip() for line in s.splitlines() if line.strip()]
    bulletish = sum(
        1 for line in lines
        if line.startswith(("-", "*")) or re.match(r"^\d+[\.)]\s+", line)
    )
    if bulletish:
        return bulletish
    separators = max(s.count(";"), s.count("\n\n"))
    if separators:
        return separators + 1
    if " one risk factor" in s.lower() or " a risk factor" in s.lower():
        return 1
    return 1


_SecAgenticRAGWorkflowFinalBase = SecAgenticRAGWorkflow


class SecAgenticRAGWorkflow(_SecAgenticRAGWorkflowFinalBase):
    @step
    async def context_evaluator(self, _ctx: Context, ev: RetrievedEvent) -> GenerateEvent | RetrieveEvent:
        state = ev.state
        retrieval_signature = (
            tuple(state.get("retrieved_doc_names", [])[:10]),
            len(state.get("retrieved_chunks", []) or []),
        )

        if state.get("retrieval_sanity_failed", False):
            relevant = False
        else:
            context = format_retrieved_context(
                state.get("retrieved_chunks", []),
                max_chunks=CONFIG["control_max_context_chunks"],
                max_chars=CONFIG["control_max_context_chars"],
            )
            judged = invoke_structured(
                context_eval_llm,
                ContextEvalOutput,
                CONTEXT_EVAL_SYSTEM,
                f"Question:\n{state['question']}\n\nContext:\n{context}",
                ContextEvalOutput(relevant=True, reason="fallback"),
            )
            relevant = judged.relevant

        state["context_eval_relevant"] = relevant
        prev_signature = state.get("last_retrieval_signature")
        state["last_retrieval_signature"] = retrieval_signature

        if relevant:
            return GenerateEvent(state=state)

        retries = state.get("context_retries", 0)
        identical_retry = bool(
            CONFIG.get("avoid_identical_retrieval_retries", True)
            and prev_signature is not None
            and prev_signature == retrieval_signature
        )
        has_context = bool(state.get("retrieved_chunks"))
        if retries < CONFIG["max_context_retries"] and not identical_retry and not has_context:
            state["context_retries"] = retries + 1
            return RetrieveEvent(state=state)
        if identical_retry:
            state["context_retry_skipped_reason"] = "identical_retrieval_signature"

        return GenerateEvent(state=state)

    @step
    async def generator(self, _ctx: Context, ev: GenerateEvent) -> CritiqueEvent:
        state = ev.state
        context = format_retrieved_context(
            state.get("retrieved_chunks", []),
            max_chunks=CONFIG["generator_max_context_chunks"],
            max_chars=CONFIG["generator_max_context_chars"],
        )
        extra_instruction = ""
        if _looks_enumeration_question(state.get("question", "")):
            extra_instruction = (
                "\n\nExtra instruction: This is a multi-item question. Synthesize all distinct supported items from the retrieved context, deduplicate them, and present them as concise bullet points. Do not stop after the first supported item."
            )

        answer = invoke_text(
            generator_llm,
            GENERATOR_SYSTEM,
            f"Question:\n{state['question']}\n\nRetrieved context:\n{context}{extra_instruction}",
            fallback_text="",
        ).strip()

        state["generated_answer"] = answer
        state["final_answer"] = answer
        return CritiqueEvent(state=state)

    @step
    async def critic(self, _ctx: Context, ev: CritiqueEvent) -> JudgeEvent | RepairEvent | DriftDetectEvent:
        state = ev.state

        if state.get("is_repair_retrieval", False):
            return JudgeEvent(state=state)

        context = format_retrieved_context(
            state.get("retrieved_chunks", []),
            max_chunks=CONFIG["control_max_context_chunks"],
            max_chars=CONFIG["control_max_context_chars"],
        )

        result = invoke_structured(
            "agent",
            CriticOutput,
            CRITIC_SYSTEM,
            (
                f"Question:\n{state['question']}\n\n"
                f"Context:\n{context}\n\n"
                f"Answer:\n{state.get('generated_answer', '')}"
            ),
            CriticOutput(decision="accept", reason="fallback"),
            task_name="critic",
        )

        answer_text = str(state.get("generated_answer", "")).strip()
        if result.decision == "accept" and _looks_enumeration_question(state.get("question", "")):
            item_count = _enumerated_item_count(answer_text)
            if item_count < 2 and compute_decision_from_text(answer_text) != "refuse":
                result = CriticOutput(
                    decision="repair",
                    reason="Answer appears incomplete for a multi-item question; retrieve or synthesize more supported items.",
                )

        state["critic_decision"] = result.decision
        state["critic_reason"] = result.reason

        if result.decision == "repair":
            return RepairEvent(state=state)

        if result.decision == "insufficient_data":
            state["final_answer"] = (
                "Insufficient data: The required information could not be found in the "
                f"retrieved SEC filings. ({result.reason})"
            )
            has_retrieved_context = bool(state.get("retrieved_chunks"))
            should_drift = bool(CONFIG.get("enable_drift_and_scrape", False))
            if CONFIG.get("drift_only_on_empty_context", True):
                should_drift = should_drift and (not has_retrieved_context or state.get("retrieval_sanity_failed", False))
            if should_drift:
                return DriftDetectEvent(state=state)
            return JudgeEvent(state=state)

        return JudgeEvent(state=state)


async def run_agentic_rag(
    question: str,
    index: CorpusIndex,
    golden_answer: Optional[str] = None,
    advanced_output: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    wf = SecAgenticRAGWorkflow(timeout=float(CONFIG.get("workflow_timeout_sec", 240)), verbose=True)
    result = await wf.run(
        question=question,
        index=index,
        golden_answer=golden_answer,
        advanced_output=advanced_output,
    )
    out = dict(result)
    out["output_source"] = (
        "seeded_prior_advanced"
        if (advanced_output or {}).get("output_source") == "prior_advanced_results"
        else "live_agentic_rag"
    )
    out["seeded_from_prior_advanced"] = bool(
        (advanced_output or {}).get("output_source") == "prior_advanced_results"
    )
    return out


# Quick demo
# question = "How did Apple's gross margin percentage change between fiscal year 2023 and fiscal year 2024?"
# golden_answer = "Apple's gross margin rose from 44.1% in FY2023 to 46.6% in FY2024."
# result = await run_agentic_rag(question, global_index, golden_answer=golden_answer)
#
# print("Question:", question)
# print("Final answer:\n", result.get("final_answer", ""))
# print("Critic decision:", result.get("critic_decision"))
# print("Judge score:", result.get("judge_score"), "| Claims covered:", result.get("judge_claims_covered"))
# print("Repair used:", result.get("repair_used"), "| Repair retrieval count:", result.get("repair_retrieval_count"))

In [15]:
# Optional single-question smoke test (manual debugging only; not part of batch evaluation)
if not CONFIG.get("run_smoke_test", True):
    print("[Smoke Test] Skipped. Set CONFIG['run_smoke_test'] = True to enable.")
else:
    question = "What are the latest risk factors for NVIDIA?"
    golden_answer = (
        "NVIDIA's latest risk factors include intense competition, supply chain and manufacturing constraints, "
        "rapid technological change, export controls and geopolitical restrictions, and dependence on key customers/suppliers."
    )
    result = await run_agentic_rag(question, global_index, golden_answer=golden_answer)

    print("Question:", question)
    print("Final answer:\n", result.get("final_answer", ""))
    print("Critic decision:", result.get("critic_decision"))
    print("Judge score:", result.get("judge_score"), "| Claims covered:", result.get("judge_claims_covered"))
    print("Judge reason:", result.get("judge_reason"))
    print("Repair used:", result.get("repair_used"), "| Repair retrieval count:", result.get("repair_retrieval_count"))
    print("Drift triggered:", result.get("drift_triggered"), "| Ingestion ran:", result.get("ingestion_just_ran"))
    print("Retrieved docs:", result.get("retrieved_doc_names", []))

[Smoke Test] Skipped. Set CONFIG['run_smoke_test'] = True to enable.


In [16]:
# Ensure required retry keys exist for workflow execution paths.
CONFIG.setdefault("max_context_retries", 2)
CONFIG.setdefault("max_repair_retrievals", 1)
CONFIG.setdefault("eval_checkpoint_every_n", 5)

print("Runtime evaluation config:", {
    "use_pilot": CONFIG.get("use_pilot"),
    "full_n_questions": CONFIG.get("full_n_questions"),
    "eval_split": CONFIG.get("eval_split"),
    "eval_checkpoint_every_n": CONFIG.get("eval_checkpoint_every_n"),
    "max_context_retries": CONFIG.get("max_context_retries"),
    "max_repair_retrievals": CONFIG.get("max_repair_retrievals"),
})

Runtime evaluation config: {'use_pilot': False, 'full_n_questions': 80, 'eval_split': 'test', 'eval_checkpoint_every_n': 5, 'max_context_retries': 2, 'max_repair_retrievals': 1}


In [17]:
# ---------------------------------------------------------------------------
# CrewAI-style evaluation export for LlamaIndex (robust full-run mode)
# ---------------------------------------------------------------------------

from collections import Counter
from tqdm.auto import tqdm


def _tokenize(text: str) -> List[str]:
    return re.sub(r"[^\w\s]", "", str(text).lower()).split()


def token_f1_score(pred: str, ref: str) -> float:
    if not pred or not ref:
        return 0.0
    pred_toks = _tokenize(pred)
    ref_toks = _tokenize(ref)
    if not pred_toks or not ref_toks:
        return 0.0
    common = sum((Counter(pred_toks) & Counter(ref_toks)).values())
    if common == 0:
        return 0.0
    precision = common / len(pred_toks)
    recall = common / len(ref_toks)
    return 2 * precision * recall / (precision + recall)


def numerical_accuracy_score(pred: str, ref: str) -> Optional[float]:
    nums_ref = [float(n.replace(",", "")) for n in re.findall(r"-?\d[\d,]*\.?\d*", str(ref))]
    nums_pred = [float(n.replace(",", "")) for n in re.findall(r"-?\d[\d,]*\.?\d*", str(pred))]
    if not nums_ref:
        return None
    hits = sum(1 for r in nums_ref if any(abs(p - r) / (abs(r) + 1e-9) < 0.01 for p in nums_pred))
    return hits / len(nums_ref)


def compute_decision_from_text(answer_text: str) -> str:
    lower = str(answer_text).lower()
    refusal_phrases = [
        "insufficient", "not contain", "not available", "cannot find",
        "don't have", "no information", "not provided", "unable to",
        "not enough", "not present", "not found", "insufficient data",
    ]
    return "refuse" if any(p in lower for p in refusal_phrases) else "answer"


JUDGE_FAITHFULNESS_SYSTEM = (
    "You are a strict financial QA judge. Score faithfulness of candidate_answer to retrieved_context from 0.0 to 1.0. "
    "Score 1.0 only if claims are fully supported by context. "
    "Return valid JSON only with keys score, claims_covered, reason."
)


def llm_judge_faithfulness(question: str, context: str, candidate_answer: str) -> tuple:
    judged = invoke_structured(
        "judge",
        JudgeOutput,
        JUDGE_FAITHFULNESS_SYSTEM,
        (
            f"Question:\n{question}\n\n"
            f"Retrieved context:\n{context[:int(CONFIG.get('judge_max_context_chars', 4000))]}\n\n"
            f"Candidate answer:\n{candidate_answer}"
        ),
        JudgeOutput(score=0.0, claims_covered=0.0, reason="Judge fallback"),
        task_name="judge_faithfulness",
    )
    model_used = _preferred_models_by_task.get("judge_faithfulness", "unknown") if "_preferred_models_by_task" in globals() else "unknown"
    return (
        max(0.0, min(1.0, float(judged.score))),
        max(0.0, min(1.0, float(judged.claims_covered))),
        judged.reason,
        model_used,
    )


# ---------- Data loading + deterministic evaluation selection ----------
raw_eval_df = pd.read_csv(CONFIG["sec_eval_csv_path"])
raw_eval_df = raw_eval_df[raw_eval_df["question_id"].notna()].copy()
raw_eval_df["question_id"] = raw_eval_df["question_id"].astype(str).str.strip()
raw_eval_df = raw_eval_df[raw_eval_df["question_id"] != ""].copy()

full_eval_df = raw_eval_df.rename(columns={
    "question_id": "id",
    "expected_answer": "reference_answer",
}).copy()

for col in ("expected_decision", "question_type", "company", "reference_answer", "split"):
    if col in full_eval_df.columns:
        full_eval_df[col] = full_eval_df[col].fillna("").astype(str).str.strip()

for col in ("ticker", "form_type"):
    if col in full_eval_df.columns:
        full_eval_df[col] = full_eval_df[col].fillna("").astype(str).str.strip()

if "id" in full_eval_df.columns:
    full_eval_df["id"] = full_eval_df["id"].apply(
        lambda x: f"SEC_{int(float(x)):03d}" if str(x).replace(".", "", 1).isdigit() else str(x)
    )

if "split" in full_eval_df.columns:
    full_eval_df["split"] = full_eval_df["split"].astype(str).str.strip().str.lower()

target_split = str(CONFIG.get("eval_split", "test")).strip().lower()
eval_source_df = full_eval_df[
    full_eval_df.get("split", pd.Series("", index=full_eval_df.index)).isin([target_split])
].copy()

if eval_source_df.empty:
    raise ValueError(
        f"No rows found for eval split={target_split!r}. Available splits: {full_eval_df.get('split', pd.Series()).value_counts(dropna=False).to_dict()}"
    )

requested_n_questions = int(CONFIG.get("full_n_questions", 80))

eval_df = (
    eval_source_df
    .sort_values("id")
    .head(requested_n_questions)
    .reset_index(drop=True)
)

split_counts = full_eval_df.get("split", pd.Series("", index=full_eval_df.index)).value_counts(dropna=False).to_dict()
print(f"[Pre-run] split counts (all rows): {split_counts}")
print(f"[Pre-run] eval_source_df rows for split='{target_split}': {len(eval_source_df)}")
print(f"[Pre-run] eval_df rows selected: {len(eval_df)} | requested={requested_n_questions}")

assert requested_n_questions > 0, f"full_n_questions must be > 0, got {requested_n_questions}"
assert len(eval_source_df) >= requested_n_questions, (
    f"Expected at least {requested_n_questions} rows in split={target_split!r}, found {len(eval_source_df)}"
)
assert len(eval_df) == requested_n_questions, (
    f"Expected eval_df length {requested_n_questions}, got {len(eval_df)}"
)


# ---------- Robust evaluation loop with per-question fault tolerance ----------
out_dir = PROJECT_ROOT / "Framework - Llama" / "results"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "eval_results_llamaindex.csv"
out_path_legacy = out_dir / "llamaindex_eval_results.csv"
checkpoint_every = int(CONFIG.get("eval_checkpoint_every_n", 5))
checkpoint_path = out_dir / "llamaindex_eval_results.checkpoint.csv"


def _write_checkpoint(rows: List[Dict[str, Any]], path: Path) -> None:
    if not rows:
        return
    checkpoint_df = pd.DataFrame(rows)
    checkpoint_export_df = checkpoint_df.drop(columns=["retrieved_chunks"], errors="ignore")
    checkpoint_export_df.to_csv(path, index=False)


def _mean_with_count(series: pd.Series) -> tuple:
    numeric = pd.to_numeric(series, errors="coerce")
    count = int(numeric.notna().sum())
    return (None if count == 0 else float(numeric.mean()), count)


def _format_metric_line(label: str, value: Optional[float], count: int) -> str:
    value_text = "N/A" if value is None else f"{value:.4f}"
    return f"  {label:<17}: {value_text:<7}  (n={count})"


def print_llamaindex_eval_summary(results_df: pd.DataFrame, title: str = "AGENTIC RAG — EVALUATION RESULTS") -> None:
    results_subset = results_df.copy()
    print("=" * 60)
    print(f" {title}")
    print("=" * 60)

    scored_c = results_subset[results_subset["llm_correctness_score"].notna()] if "llm_correctness_score" in results_subset.columns else results_subset.iloc[0:0]
    scored_f = results_subset[results_subset["llm_faithfulness_score"].notna()] if "llm_faithfulness_score" in results_subset.columns else results_subset.iloc[0:0]
    print(f"\nTotal questions    : {len(results_subset)}")
    print(f"Correctness judged : {len(scored_c)}")
    print(f"Faithfulness judged: {len(scored_f)}")

    print("\nOverall metrics:")
    for col, label in [
        ("llm_correctness_score",  "LLM Correctness  "),
        ("llm_claims_covered",     "Claims Covered   "),
        ("llm_faithfulness_score", "LLM Faithfulness "),
        ("token_f1",               "Token F1         "),
        ("numerical_accuracy",     "Numerical Accuracy"),
        ("decision_accuracy",      "Decision Accuracy"),
    ]:
        if col not in results_subset.columns:
            continue
        vals = results_subset[col].dropna()
        if len(vals):
            print(f"  {label}: {vals.mean():.4f}  (n={len(vals)})")

    print("\nBy question_type:")
    type_cols = ["llm_correctness_score", "llm_faithfulness_score", "token_f1", "decision_accuracy"]
    avail = [c for c in type_cols if c in results_subset.columns]
    if avail and "question_type" in results_subset.columns and len(results_subset):
        print(results_subset.groupby("question_type")[avail].mean().round(3).to_string())
    else:
        print("(No question_type breakdown available)")

    if "latency_sec" in results_subset.columns:
        lat = results_subset["latency_sec"].dropna()
        if len(lat):
            print(f"\nLatency: mean={lat.mean():.1f}s  median={lat.median():.1f}s  max={lat.max():.1f}s")

    token_cols = [
        "tokens_generate_input", "tokens_generate_output",
        "tokens_judge_corr_input", "tokens_judge_corr_output",
        "tokens_judge_faith_input", "tokens_judge_faith_output",
        "tokens_total_input", "tokens_total_output",
    ]
    present_token_cols = [c for c in token_cols if c in results_subset.columns]
    if present_token_cols or "est_cost_usd" in results_subset.columns:
        print("\nToken & cost summary:")
        for col in present_token_cols:
            print(f"  {col:35s}: {int(results_subset[col].fillna(0).sum()):,}")
        if "est_cost_usd" in results_subset.columns:
            print(f"  {'Total estimated cost (USD)':35s}: ${results_subset['est_cost_usd'].fillna(0).sum():.4f}")


async def run_llamaindex_evaluation(df: pd.DataFrame) -> pd.DataFrame:
    all_results: List[Dict[str, Any]] = []
    sleep_sec = max(1.0, float(CONFIG.get("inter_question_sleep_sec", 1.5)))
    prior_advanced_cache = load_prior_advanced_results(df)

    for i, row in tqdm(df.iterrows(), total=len(df), desc="LlamaIndex Eval"):
        question = str(row.get("question", "")).strip()
        reference_answer = str(row.get("reference_answer", "")).strip()
        question_id = row.get("question_id", row.get("id", i))
        question_type = row.get("question_type", "unknown")
        company = row.get("company", "")
        ticker = row.get("ticker", "")
        form_type = row.get("form_type", "")
        filing_year = row.get("filing_year", None)
        expected_decision = str(row.get("expected_decision", "answer")).lower().strip()

        try:
            # Mirror LangGraph seeding: reuse prior advanced output as agentic input, not as agentic result.
            prior_row = get_prior_advanced_row(row, prior_advanced_cache)
            advanced_output = None
            if prior_row is not None:
                print(f"[Cache Hit] {question_id} | source: prior_advanced_results_csv | mode: seeded_agentic")
                prior_copy = prior_row.copy()
                prior_copy["_source_path"] = prior_advanced_cache.get("path")
                advanced_output = build_advanced_output_from_prior(row, prior_copy, prior_advanced_cache.get("path", ""))
            start = time.time()
            result = await run_agentic_rag(
                question,
                global_index,
                golden_answer=reference_answer or None,
                advanced_output=advanced_output,
            )
            latency_sec = time.time() - start

            final_answer = str(result.get("final_answer", ""))
            generated_answer = str(result.get("generated_answer", ""))
            retrieved_chunks = result.get("retrieved_chunks", []) or []
            retrieved_doc_names = result.get("retrieved_doc_names", []) or []

            t_f1 = token_f1_score(final_answer, reference_answer) if reference_answer else None
            num_acc = numerical_accuracy_score(final_answer, reference_answer) if reference_answer else None
            predicted_decision = compute_decision_from_text(final_answer)
            decision_accuracy = 1.0 if predicted_decision == expected_decision else 0.0

            context_str = format_retrieved_context(
                retrieved_chunks,
                max_chunks=int(CONFIG.get("judge_max_context_chunks", 3)),
                max_chars=int(CONFIG.get("judge_max_context_chars", 4000)),
            )

            c_score = result.get("judge_score", result.get("llm_correctness_score", None))
            c_cov = result.get("judge_claims_covered", result.get("llm_claims_covered", None))
            c_reason = result.get("judge_reason", result.get("llm_correctness_reason", None))
            c_model = _preferred_models_by_task.get("judge", "unknown") if "_preferred_models_by_task" in globals() else "unknown"

            precomputed_f_score = result.get("llm_faithfulness_score", None)
            precomputed_f_reason = result.get("llm_faithfulness_reason", None)
            if precomputed_f_score is not None:
                f_score = float(precomputed_f_score)
                f_reason = precomputed_f_reason or "Loaded from prior advanced results"
                f_model = result.get("model_judge_faithfulness", "prior_advanced_results")
            elif context_str.strip():
                try:
                    f_score, _, f_reason, f_model = llm_judge_faithfulness(question, context_str, final_answer)
                except Exception as judge_exc:
                    f_score, f_reason, f_model = 0.0, f"Judge failed: {judge_exc}", "unknown"
            else:
                f_score, f_reason, f_model = None, "Skipped: no retrieved context available for faithfulness judging.", "unknown"

            model_snapshot = {
                "planner": _preferred_models_by_task.get("planner", "unknown") if "_preferred_models_by_task" in globals() else "unknown",
                "generator": _preferred_models_by_task.get("generator", "unknown") if "_preferred_models_by_task" in globals() else "unknown",
                "critic": _preferred_models_by_task.get("critic", "unknown") if "_preferred_models_by_task" in globals() else "unknown",
            }

            result_row = {
                "question_id": question_id,
                "question": question,
                "question_type": question_type,
                "company": company,
                "ticker": ticker,
                "form_type": form_type,
                "filing_year": filing_year,
                "reference_answer": reference_answer,
                "expected_decision": expected_decision,
                "final_answer": final_answer,
                "generated_answer": generated_answer,
                "pipeline": "llamaindex_agentic_rag",
                "retrieved_doc_names": ",".join(map(str, retrieved_doc_names)),
                "needs_decomposition": result.get("needs_decomposition", False),
                "critic_decision": result.get("critic_decision", ""),
                "repair_used": result.get("repair_used", False),
                "latency_sec": latency_sec,
                "token_f1": t_f1,
                "numerical_accuracy": num_acc,
                "decision_accuracy": decision_accuracy,
                "predicted_decision": predicted_decision,
                "llm_correctness_score": c_score,
                "llm_claims_covered": c_cov,
                "llm_correctness_reason": c_reason,
                "llm_faithfulness_score": f_score,
                "llm_faithfulness_reason": f_reason,
                "model_planner": result.get("model_planner", model_snapshot.get("planner")),
                "model_generator": result.get("model_generator", model_snapshot.get("generator")),
                "model_critic": result.get("model_critic", model_snapshot.get("critic")),
                "model_judge_correctness": c_model,
                "model_judge_faithfulness": f_model,
                "tokens_crew_input": 0,
                "tokens_crew_output": 0,
                "tokens_judge_corr_input": 0,
                "tokens_judge_corr_output": 0,
                "tokens_judge_faith_input": 0,
                "tokens_judge_faith_output": 0,
                "tokens_total_input": 0,
                "tokens_total_output": 0,
                "est_cost_usd": 0.0,
                "retrieved_chunks": retrieved_chunks,
            }
            all_results.append(result_row)

        except Exception as eval_exc:
            print(f"[ERROR] Question {question_id} failed: {eval_exc}")
            all_results.append({
                "question_id": question_id,
                "question": question,
                "question_type": question_type,
                "company": company,
                "ticker": ticker,
                "form_type": form_type,
                "filing_year": filing_year,
                "reference_answer": reference_answer,
                "expected_decision": expected_decision,
                "final_answer": "",
                "generated_answer": "",
                "pipeline": "llamaindex_agentic_rag",
                "retrieved_doc_names": "",
                "needs_decomposition": False,
                "critic_decision": "error",
                "repair_used": False,
                "latency_sec": None,
                "token_f1": None,
                "numerical_accuracy": None,
                "decision_accuracy": 0.0,
                "predicted_decision": "error",
                "llm_correctness_score": 0.0,
                "llm_claims_covered": 0.0,
                "llm_correctness_reason": f"Evaluation failed: {eval_exc}",
                "llm_faithfulness_score": 0.0,
                "llm_faithfulness_reason": f"Evaluation failed: {eval_exc}",
                "model_planner": "unknown",
                "model_generator": "unknown",
                "model_critic": "unknown",
                "model_judge_correctness": "unknown",
                "model_judge_faithfulness": "unknown",
                "tokens_crew_input": 0,
                "tokens_crew_output": 0,
                "tokens_judge_corr_input": 0,
                "tokens_judge_corr_output": 0,
                "tokens_judge_faith_input": 0,
                "tokens_judge_faith_output": 0,
                "tokens_total_input": 0,
                "tokens_total_output": 0,
                "est_cost_usd": 0.0,
                "retrieved_chunks": [],
            })

        if checkpoint_every > 0 and len(all_results) % checkpoint_every == 0:
            _write_checkpoint(all_results, checkpoint_path)
            print(f"[Checkpoint] Saved {len(all_results)} rows -> {checkpoint_path}")

        time.sleep(sleep_sec)

    return pd.DataFrame(all_results)


eval_write_failed = False
eval_write_failure_reason = ""

print(f"Running LlamaIndex evaluation on {len(eval_df)} questions...")
results_df = await run_llamaindex_evaluation(eval_df)

if checkpoint_path.exists():
    print(f"Checkpoint file exists at: {checkpoint_path}")

drop_cols = ["retrieved_chunks"]
export_df = results_df.drop(columns=[c for c in drop_cols if c in results_df.columns], errors="ignore")

try:
    export_df.to_csv(out_path, index=False)
    export_df.to_csv(out_path_legacy, index=False)
except Exception as export_exc:
    eval_write_failed = True
    eval_write_failure_reason = str(export_exc)
    print(f"[WARN] Normal export failed: {export_exc}")
else:
    print(f"\nResults saved -> {out_path}")
    print(f"Results also saved -> {out_path_legacy}")

print(f"Final row count: {len(export_df)}")
print()
print_llamaindex_eval_summary(export_df)
results_df.head(10)

[Pre-run] split counts (all rows): {'test': 80, 'dev': 20}
[Pre-run] eval_source_df rows for split='test': 80
[Pre-run] eval_df rows selected: 80 | requested=80
Running LlamaIndex evaluation on 80 questions...
✓ Loaded prior advanced results: C:\Users\jeeey\GenAI-with-LLMs\baseline_advanced_rag\results\advanced_results.csv | rows=100 | matches_by_id=80/80 | matches_by_question=80/80


LlamaIndex Eval:   0%|          | 0/80 [00:00<?, ?it/s]

[Cache Hit] SEC_006 | source: prior_advanced_results_csv | mode: seeded_agentic
[tick] add: StartEvent(question="What was Nvidia's gross margin percentage in fisca...", index=<shared_retriever.CorpusIndex object at 0x0000020360897B10>, golden_answer="Nvidia's gross margin was 72.7% in FY2...
[query_planning:0] started from StartEvent
  [Planner] single | 1 sub-query
[query_planning:0] complete with PlanEvent
[tick] add: PlanEvent(state={13 keys})
[hybrid_retriever:0] started from PlanEvent
    [DenseDebug] query='NVDA Nvidia s gross margin percentage in fiscal year 2024' where={'$and': [{'ticker': {'$eq': 'NVDA'}}, {'filing_year': {'$eq': 2024}}]} filtered_hits=0 broad_hits=60 fallback_mode=relax_year local_filtered_hits=15 returned=15 unresolved_sample=[]
    [Retrieval] pool=51 (bm25=15 dense=15 adj=25) → rerank top-7
[hybrid_retriever:0] complete with RetrievedEvent
[tick] add: RetrievedEvent(state={15 keys})
[context_evaluator:0] started from RetrievedEvent
[context_evaluator:0] co

Retrying llama_index.llms.google_genai.base.GoogleGenAI._achat in 1.0 seconds as it raised ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}.


[Cache Hit] SEC_057 | source: prior_advanced_results_csv | mode: seeded_agentic
[tick] add: StartEvent(question='In Nvidia’s fiscal year 2025 10-K, how much larger...', index=<shared_retriever.CorpusIndex object at 0x0000020360897B10>, golden_answer='The interest-rate sensitivity was $109...
[query_planning:0] started from StartEvent
  [Planner] single | 1 sub-query
[query_planning:0] complete with PlanEvent
[tick] add: PlanEvent(state={13 keys})
[hybrid_retriever:0] started from PlanEvent
    [DenseDebug] query='NVDA In NVIDIA s fiscal year 2025 10-K, find the quantified impact of a 0.5% int' where={'$and': [{'ticker': {'$eq': 'NVDA'}}, {'filing_year': {'$eq': 2025}}]} filtered_hits=0 broad_hits=60 fallback_mode=relax_year local_filtered_hits=15 returned=15 unresolved_sample=[]
    [Retrieval] pool=80 (bm25=15 dense=15 adj=56) → rerank top-7
[hybrid_retriever:0] complete with RetrievedEvent
[tick] add: RetrievedEvent(state={15 keys})
[context_evaluator:0] started from RetrievedEvent
[

,question_id,question,question_type,company,ticker,form_type,filing_year,reference_answer,expected_decision,final_answer,...,tokens_crew_input,tokens_crew_output,tokens_judge_corr_input,tokens_judge_corr_output,tokens_judge_faith_input,tokens_judge_faith_output,tokens_total_input,tokens_total_output,est_cost_usd,retrieved_chunks
0,SEC_006,What was Nvidia's gross margin percentage in f...,extractive,NVIDIA,NVDA,10-K,2024.0,"Nvidia's gross margin was 72.7% in FY2024, up ...",answer,The retrieved context provides the following g...,...,0,0,0,0,0,0,0,0,0.0,[RetrievedChunk(doc_name='NVDA_10-Q_2024-05-29...
1,SEC_007,How many employees did Nvidia report at the en...,extractive,NVIDIA,NVDA,10-K,2025.0,"As of the end of fiscal year 2025, Nvidia had ...",answer,The retrieved context does not contain informa...,...,0,0,0,0,0,0,0,0,0.0,[RetrievedChunk(doc_name='NVDA_10-Q_2025-05-28...
2,SEC_008,What was JPMorgan's CET1 capital ratio at the ...,extractive,JPMorgan Chase,JPM,10-K,2023.0,JPMorgan's Standardized and Advanced CET1 capi...,answer,The retrieved context does not contain JPMorga...,...,0,0,0,0,0,0,0,0,0.0,[RetrievedChunk(doc_name='JPM_10-Q_2023-05-03'...
3,SEC_009,Which federal regulator supervises JPMorgan Ch...,extractive,JPMorgan Chase,JPM,10-K,2024.0,JPMorgan Chase is supervised and regulated by ...,answer,"JPMorgan Chase & Co., as a bank holding compan...",...,0,0,0,0,0,0,0,0,0.0,[RetrievedChunk(doc_name='JPM_10-K_2025-02-14'...
4,SEC_010,What were Bank of America's Total liabilities ...,extractive,Bank of America,BAC,10-Q,2025.0,Bank of America's total liabilities as of Sept...,answer,Insufficient data: The required information co...,...,0,0,0,0,0,0,0,0,0.0,[RetrievedChunk(doc_name='BAC_10-Q_2025-07-31'...
5,SEC_011,What interest rate risk does Bank of America d...,extractive,Bank of America,BAC,10-K,2024.0,Bank of America manages interest rate risk thr...,answer,The retrieved context does not contain Bank of...,...,0,0,0,0,0,0,0,0,0.0,[RetrievedChunk(doc_name='BAC_10-Q_2025-07-31'...
6,SEC_012,What was Berkshire Hathaway’s net earnings att...,extractive,Berkshire Hathaway,BRK-B,10-K,2025.0,Berkshire Hathaway's net earnings attributable...,answer,Insufficient data: The required information co...,...,0,0,0,0,0,0,0,0,0.0,[RetrievedChunk(doc_name='BRK-B_10-Q_2024-08-0...
7,SEC_013,How much did Berkshire Hathaway pay to acquire...,extractive,Berkshire Hathaway,BRK-B,10-Q,2023.0,Berkshire Hathaway acquired an additional 41.4...,answer,Berkshire Hathaway paid approximately $8.2 bil...,...,0,0,0,0,0,0,0,0,0.0,[RetrievedChunk(doc_name='BRK-B_10-K_2026-03-0...
8,SEC_014,What were Johnson & Johnson’s total sales for ...,extractive,Johnson & Johnson,JNJ,10-Q,2025.0,Johnson & Johnson's total sales for the nine m...,answer,Johnson & Johnson's total sales for the first ...,...,0,0,0,0,0,0,0,0,0.0,[RetrievedChunk(doc_name='JNJ_10-Q_2025-10-22'...
9,SEC_015,What was Johnson & Johnson’s net earnings in t...,extractive,Johnson & Johnson,JNJ,10-K,2025.0,Johnson & Johnson's net earnings from continui...,answer,Johnson & Johnson's net earnings in the 2024 A...,...,0,0,0,0,0,0,0,0,0.0,[RetrievedChunk(doc_name='JNJ_10-K_2024-02-16'...


In [18]:
# Disk-space recovery rerun: only execute if the normal export path failed.
if not globals().get("eval_write_failed", False):
    print("[Recovery] Skipped: normal evaluation/export already completed successfully.")
else:
    checkpoint_every = 0

    def _write_checkpoint(rows: List[Dict[str, Any]], path: Path) -> None:
        return

    print(f"[Recovery] Triggered because normal export failed: {globals().get('eval_write_failure_reason', 'unknown error')}")
    print("[Recovery] Re-running evaluation without checkpoint/final CSV writes.")

    results_df = await run_llamaindex_evaluation(eval_df)
    export_df = results_df.drop(columns=["retrieved_chunks"], errors="ignore")

    print(f"[Recovery] Evaluation completed in-memory. Rows: {len(export_df)}")
    print("[Recovery] Skipped writing CSV files due to the earlier export failure.")
    print()
    print_llamaindex_eval_summary(export_df)
    export_df.head(10)

[Recovery] Skipped: normal evaluation/export already completed successfully.
